# NB13 - Native-assisted AVI pipeline improvement validation

## Purpose

This notebook tests whether native metadata and native timing/QC sidecar signals can support the existing AVI/audio Doppler pipeline while preserving clear evidence boundaries.

This notebook is the final validation bridge before package modularization. It rebuilds the validation step by step, compares each candidate native-assisted change against a reproduced AVI image-pipeline baseline, and identifies which stable helper functions are suitable for later nbdev export.

## Scope

### In scope

- Repository-relative path setup.
- Native-to-AVI mapping using linked AVI file sizes.
- Baseline AVI frame-level velocity reproduction from the NB04 V2 export.
- Native `DcmRegionPara` ROI, baseline, velocity scale, and time scale extraction.
- Native `PW_CinePartition0.bin` timing/QC sidecar exploration.
- Candidate native scale, ROI, and baseline-override calibration deltas.
- Direction-aware baseline override checks for spectra above or below the baseline.
- Candidate sweep-marker masking robustness checks.
- Candidate autocorrelation-guided beat detector checks with morphology recomputation.
- Candidate audio clipping QC.
- Audio/native HR agreement as an experimental QC comparison.
- Evidence-stage classification for each candidate improvement.
- Identification of stable reusable helpers for later nbdev modularization.

### Future nbdev modularization boundary

Selected stable helper functions may include nbdev export directives such as `#| export paths`, `#| export video`, `#| export waveform`, `#| export calibration`, `#| export native_mapping`, `#| export native_metadata`, `#| export native_qc`, or `#| export audio_qc`.

These directives are preparation for later package export. They do not by themselves make candidate behavior a pipeline default.

Validation cells, full-frame comparison cells, exploratory candidate checks, and evidence-stage checkpoint logic remain notebook evidence unless explicitly promoted later.

### Out of scope

- Editing NB01 through NB12.
- Treating candidate deltas as accepted defaults.
- Clinical validation.
- Native spectrogram recovery.
- Native velocity envelope recovery.
- Native PSV, EDV, RI, PI, or VTI recovery.
- Production-ready Doppler measurement claims.
- Package release, README generation, or nbdev build execution before the NB13 evidence checkpoint is reviewed.

## Interpretation Boundary

This notebook evaluates candidate native-assisted support signals for the AVI/audio pipeline.

Native metadata and native timing/QC outputs are treated as experimental, metadata-derived or sidecar signals unless independently reproduced and compared against current project evidence.

This notebook does not claim calibrated clinical measurements, decoded native spectrograms, native velocity envelopes, or validated diagnostic outputs.


## Repository path setup


In [1]:
#| export paths
from pathlib import Path


def find_repo_root(start=None, markers=("ultrasound_recordings", "feature_exports")):
    """
    This function resolves the DopplerLab repository root by walking upward
    from a starting directory until all marker paths are found.
    """
    if start is None:
        current = Path.cwd().resolve()
    else:
        current = Path(start).resolve()

    for candidate in (current, *current.parents):
        if all((candidate / marker).exists() for marker in markers):
            return candidate

    marker_text = ", ".join(markers)
    raise FileNotFoundError(
        f"Could not locate DopplerLab repo root from {current}. "
        f"Required markers: {marker_text}"
    )


def dopplerlab_project_paths(root=None):
    """
    This function builds standard DopplerLab project paths used by
    validation notebooks.
    """
    repo_root = find_repo_root() if root is None else Path(root).resolve()

    return {
        "root": repo_root,
        "recordings_dir": repo_root / "ultrasound_recordings",
        "feature_exports_dir": repo_root / "feature_exports",
        "native_batch_dir": repo_root / "ultrasound_recordings" / "batch_2026_06_13_native",
        "avi_batch_dir": repo_root / "ultrasound_recordings" / "batch_2026_06_13",
        "nb04_v2_frame_csv": repo_root / "feature_exports" / "nb04_v2_batch_2026_06_13" / "nb04_v2_frame_level_velocity_features.csv",
    }

In [2]:
paths = dopplerlab_project_paths()

path_checks = [
    ("repo root", paths["root"]),
    ("ultrasound_recordings/", paths["recordings_dir"]),
    ("feature_exports/", paths["feature_exports_dir"]),
    ("native batch folder", paths["native_batch_dir"]),
    ("AVI batch folder", paths["avi_batch_dir"]),
    ("NB04 V2 frame CSV", paths["nb04_v2_frame_csv"]),
]

print("Resolved DopplerLab paths")
print("-" * 88)

missing = []

for label, path in path_checks:
    exists = path.exists()
    status = "OK" if exists else "MISSING"
    print(f"{label:28} {status:8} {path}")
    if not exists:
        missing.append((label, path))

print("-" * 88)
print(f"missing required paths: {len(missing)}")

if missing:
    missing_text = "\n".join(f"- {label}: {path}" for label, path in missing)
    raise FileNotFoundError("Required NB13 inputs are missing:\n" + missing_text)

Resolved DopplerLab paths
----------------------------------------------------------------------------------------
repo root                    OK       D:\code\DopplerLab
ultrasound_recordings/       OK       D:\code\DopplerLab\ultrasound_recordings
feature_exports/             OK       D:\code\DopplerLab\feature_exports
native batch folder          OK       D:\code\DopplerLab\ultrasound_recordings\batch_2026_06_13_native
AVI batch folder             OK       D:\code\DopplerLab\ultrasound_recordings\batch_2026_06_13
NB04 V2 frame CSV            OK       D:\code\DopplerLab\feature_exports\nb04_v2_batch_2026_06_13\nb04_v2_frame_level_velocity_features.csv
----------------------------------------------------------------------------------------
missing required paths: 0


## Step 3 - Native-to-AVI mapping from linked AVI file sizes

### What this tests

This section tests whether each native recording can be mapped to one NB04 AVI recording by matching the byte size of its `linked_avi/*.avi` file to the AVI files in the batch recording folder.

### Why this matters

Native metadata and native timing/QC signals are stored under native recording IDs, while NB04 frame-level outputs use AVI recording names. A file-derived mapping creates the bridge between those namespaces without depending on a precomputed mapping report.

### Interpretation boundary

This mapping only supports file identity by linked AVI byte size. It does not validate native metadata, Doppler waveform extraction, audio features, or clinical measurements. Duplicate-size matches are treated as ambiguous rather than silently accepted.

In [3]:
#| export native_mapping
import pandas as pd
from pathlib import Path


def build_native_to_avi_mapping(native_batch_dir, avi_batch_dir):
    """
    This function maps native recording folders to AVI recording names by
    matching the byte size of each native linked AVI file to batch AVI files.
    """
    native_batch_dir = Path(native_batch_dir)
    avi_batch_dir = Path(avi_batch_dir)

    if not native_batch_dir.exists():
        raise FileNotFoundError(f"Native batch folder does not exist: {native_batch_dir}")
    if not avi_batch_dir.exists():
        raise FileNotFoundError(f"AVI batch folder does not exist: {avi_batch_dir}")

    avi_names_by_size = {}

    for avi_path in sorted(avi_batch_dir.glob("*.avi")):
        file_size = avi_path.stat().st_size
        avi_names_by_size.setdefault(file_size, []).append(avi_path.stem)

    rows = []

    for native_recording_dir in sorted(native_batch_dir.glob("*SMP")):
        linked_avi_paths = sorted((native_recording_dir / "linked_avi").glob("*.avi"))

        if len(linked_avi_paths) == 0:
            rows.append(
                {
                    "recording_id": native_recording_dir.name,
                    "linked_avi_name": None,
                    "linked_avi_size": None,
                    "nb04_recording_name": None,
                    "candidate_count": 0,
                    "match_status": "no_linked_avi",
                }
            )
            continue

        linked_avi_path = linked_avi_paths[0]
        linked_avi_size = linked_avi_path.stat().st_size
        candidate_names = avi_names_by_size.get(linked_avi_size, [])

        if len(candidate_names) == 1:
            match_status = "matched"
            nb04_recording_name = candidate_names[0]
        elif len(candidate_names) > 1:
            match_status = "duplicate_size_ambiguous"
            nb04_recording_name = None
        else:
            match_status = "no_size_match"
            nb04_recording_name = None

        rows.append(
            {
                "recording_id": native_recording_dir.name,
                "linked_avi_name": linked_avi_path.name,
                "linked_avi_size": linked_avi_size,
                "nb04_recording_name": nb04_recording_name,
                "candidate_count": len(candidate_names),
                "match_status": match_status,
            }
        )

    return pd.DataFrame(rows)

In [4]:
native_to_avi = build_native_to_avi_mapping(
    native_batch_dir=paths["native_batch_dir"],
    avi_batch_dir=paths["avi_batch_dir"],
)

n_native = len(native_to_avi)
n_matched = int((native_to_avi["match_status"] == "matched").sum())
n_ambiguous = int((native_to_avi["match_status"] == "duplicate_size_ambiguous").sum())
n_unmatched = int(native_to_avi["match_status"].isin(["no_linked_avi", "no_size_match"]).sum())

print("Native-to-AVI mapping summary")
print("-" * 88)
print(f"native recordings:        {n_native}")
print(f"matched mappings:         {n_matched}")
print(f"ambiguous size matches:   {n_ambiguous}")
print(f"unmatched mappings:       {n_unmatched}")
print("-" * 88)

display_columns = [
    "recording_id",
    "nb04_recording_name",
    "linked_avi_size",
    "candidate_count",
    "match_status",
]

display(native_to_avi[display_columns])

if n_native != 10 or n_matched != 10 or n_ambiguous != 0 or n_unmatched != 0:
    raise ValueError(
        "Native-to-AVI mapping did not meet the expected NB13 checkpoint: "
        f"native={n_native}, matched={n_matched}, ambiguous={n_ambiguous}, unmatched={n_unmatched}"
    )

NB04_TO_RECORDING_ID = dict(
    zip(native_to_avi["nb04_recording_name"], native_to_avi["recording_id"])
)

RECORDING_ID_TO_NB04 = dict(
    zip(native_to_avi["recording_id"], native_to_avi["nb04_recording_name"])
)

Native-to-AVI mapping summary
----------------------------------------------------------------------------------------
native recordings:        10
matched mappings:         10
ambiguous size matches:   0
unmatched mappings:       0
----------------------------------------------------------------------------------------


,recording_id,nb04_recording_name,linked_avi_size,candidate_count,match_status
0,202606130411060002SMP,candidate_test_02_brachial,14816678,1,matched
1,202606130413540003SMP,candidate_test_03_brachial,12845344,1,matched
2,202606130417060004SMP,candidate_test_06_brachial,9287210,1,matched
3,202606130420260005SMP,candidate_test_08_brachial,15672588,1,matched
4,202606130422440006SMP,candidate_test_02_neck,15822224,1,matched
5,202606130426500007SMP,candidate_test_05_brachial,12638752,1,matched
6,202606130430380008SMP,candidate_test_07_brachial,10053972,1,matched
7,202606130433280009SMP,candidate_test_04_brachial_2,13273528,1,matched
8,202606130434130010SMP,candidate_test_04_brachial_1,13993442,1,matched
9,202606130437480012SMP,candidate_test_01_brachial,9847284,1,matched


## Baseline image-path helper provenance

### What this tests

This section introduces the baseline image-path helper functions used for AVI frame waveform extraction, beat morphology, manual calibration, and video frame reading.

### Why this matters

Native-assisted candidate changes can only be interpreted after the notebook reproduces the existing AVI image-pipeline behavior. These helpers provide that baseline.


In [5]:
#| export waveform
import numpy as np
import pandas as pd
import cv2
from scipy.signal import medfilt, savgol_filter, find_peaks


def extract_envelope_directional(mask, baseline, direction="auto", min_segment_length=5):
    """
    This function extracts the Doppler spectrum envelope from a binary mask.
    """
    if direction not in ["auto", "above", "below"]:
        raise ValueError("direction must be 'auto', 'above', or 'below'")
    if baseline <= 0 or baseline >= mask.shape[0] - 1:
        raise ValueError("Baseline is outside valid ROI range.")
    if min_segment_length < 1:
        raise ValueError("min_segment_length must be >= 1.")

    above_area = int(mask[:baseline, :].sum())
    below_area = int(mask[baseline + 1:, :].sum())

    if direction == "auto":
        selected_direction = "above" if above_area >= below_area else "below"
    else:
        selected_direction = direction

    envelope = np.full(mask.shape[1], np.nan)

    for x in range(mask.shape[1]):
        if selected_direction == "above":
            ys = np.where(mask[:baseline, x])[0]
        else:
            ys_local = np.where(mask[baseline + 1:, x])[0]
            ys = baseline + 1 + ys_local

        if len(ys) == 0:
            continue

        breaks = np.where(np.diff(ys) > 1)[0] + 1
        segments = np.split(ys, breaks)
        valid_segments = [segment for segment in segments if len(segment) >= min_segment_length]

        if len(valid_segments) == 0:
            continue

        if selected_direction == "above":
            envelope[x] = min(segment.min() for segment in valid_segments)
        else:
            envelope[x] = max(segment.max() for segment in valid_segments)

    if selected_direction == "above":
        velocity = baseline - envelope
    else:
        velocity = envelope - baseline

    return envelope, velocity, selected_direction


def extract_doppler_waveform(
    frame_rgb,
    x_min=80,
    x_max=640,
    y_min=230,
    y_max=510,
    threshold=80,
    direction="auto",
    min_segment_length=5,
    trim_left=10,
    trim_right=10,
    median_kernel_size=5,
    savgol_window_length=21,
    savgol_polyorder=3,
    peak_height=80,
    peak_distance=100,
):
    """
    This function extracts and smooths a Doppler waveform from a single RGB video frame.
    """
    roi = frame_rgb[y_min:y_max, x_min:x_max]
    gray = cv2.cvtColor(roi, cv2.COLOR_RGB2GRAY)
    mask = gray > threshold
    row_sums = mask.sum(axis=1)
    baseline = int(np.argmax(row_sums))

    envelope, velocity_raw, selected_direction = extract_envelope_directional(
        mask=mask,
        baseline=baseline,
        direction=direction,
        min_segment_length=min_segment_length,
    )

    if trim_left < 0 or trim_right < 0:
        raise ValueError("trim_left and trim_right must be non-negative.")
    if trim_left + trim_right >= len(envelope):
        raise ValueError("trim_left + trim_right is too large for signal length.")

    keep_slice = slice(trim_left, None) if trim_right == 0 else slice(trim_left, -trim_right)

    envelope = envelope[keep_slice]
    velocity_raw = velocity_raw[keep_slice]
    x_offset = trim_left
    valid = np.where(~np.isnan(envelope))[0]

    if len(valid) == 0:
        raise ValueError("No valid envelope points found. Check ROI, threshold, Doppler direction, min_segment_length, or trim settings.")

    envelope_interp = np.interp(np.arange(len(envelope)), valid, envelope[valid])
    velocity_raw = np.interp(np.arange(len(velocity_raw)), valid, velocity_raw[valid])

    if median_kernel_size % 2 == 0:
        median_kernel_size += 1
    velocity_med = medfilt(velocity_raw, kernel_size=median_kernel_size)

    if savgol_window_length % 2 == 0:
        savgol_window_length += 1
    if savgol_window_length >= len(velocity_med):
        savgol_window_length = len(velocity_med) - 1
        if savgol_window_length % 2 == 0:
            savgol_window_length -= 1
    if savgol_window_length <= savgol_polyorder:
        raise ValueError("savgol_window_length must be greater than savgol_polyorder.")

    velocity_smooth = savgol_filter(
        velocity_med,
        window_length=savgol_window_length,
        polyorder=savgol_polyorder,
    )

    peaks_local, peak_props = find_peaks(
        velocity_smooth,
        height=peak_height,
        distance=peak_distance,
    )
    peaks_global = peaks_local + x_offset

    return {
        "roi": roi,
        "gray": gray,
        "mask": mask,
        "baseline": baseline,
        "direction": selected_direction,
        "min_segment_length": min_segment_length,
        "trim_left": trim_left,
        "trim_right": trim_right,
        "x_offset": x_offset,
        "envelope": envelope_interp,
        "velocity_raw": velocity_raw,
        "velocity_med": velocity_med,
        "velocity_smooth": velocity_smooth,
        "peaks_local": peaks_local,
        "peaks_global": peaks_global,
        "peaks": peaks_local,
        "peak_props": peak_props,
    }

In [6]:
#| export waveform

def analyze_beats(
    result,
    min_valley_distance_from_psv=10,
    secondary_peak_search_start_after_valley=5,
    secondary_peak_search_end_after_psv=70,
    min_time_to_valley_px=15,
    max_time_to_valley_fraction=0.65,
    require_secondary_peak=True,
):
    """
    This function extracts beat-level morphology features from a Doppler waveform.
    """
    velocity = result["velocity_smooth"]
    peaks_local = result["peaks_local"]
    peaks_global = result["peaks_global"]
    x_offset = result["x_offset"]

    rows = []

    for i in range(len(peaks_local) - 1):
        psv_local = int(peaks_local[i])
        next_psv_local = int(peaks_local[i + 1])
        psv_global = int(peaks_global[i])
        next_psv_global = int(peaks_global[i + 1])

        beat_start = psv_local
        beat_end = next_psv_local

        if beat_end <= beat_start:
            continue

        cycle_length_px = int(next_psv_local - psv_local)
        psv_value = float(velocity[psv_local])

        valley_search_start = beat_start + min_valley_distance_from_psv
        valley_search_end = beat_end

        if valley_search_start >= valley_search_end:
            continue

        valley_region = velocity[valley_search_start:valley_search_end]

        if len(valley_region) == 0:
            continue

        first_valley_local = int(valley_search_start + np.argmin(valley_region))
        first_valley_global = int(first_valley_local + x_offset)
        first_valley_value = float(velocity[first_valley_local])
        time_to_valley_px = int(first_valley_local - psv_local)

        secondary_start = first_valley_local + secondary_peak_search_start_after_valley
        secondary_end = min(beat_start + secondary_peak_search_end_after_psv, beat_end)

        if secondary_start < secondary_end:
            secondary_region = velocity[secondary_start:secondary_end]
            if len(secondary_region) > 0:
                secondary_peak_local = int(secondary_start + np.argmax(secondary_region))
                secondary_peak_global = int(secondary_peak_local + x_offset)
                secondary_peak_value = float(velocity[secondary_peak_local])
            else:
                secondary_peak_local = np.nan
                secondary_peak_global = np.nan
                secondary_peak_value = np.nan
        else:
            secondary_peak_local = np.nan
            secondary_peak_global = np.nan
            secondary_peak_value = np.nan

        time_to_secondary_peak_px = (
            int(secondary_peak_local - psv_local)
            if not np.isnan(secondary_peak_local)
            else np.nan
        )

        edv_proxy = first_valley_value
        ri_proxy = (psv_value - edv_proxy) / psv_value if psv_value > 0 else np.nan
        valley_to_psv_ratio = first_valley_value / psv_value if psv_value > 0 else np.nan
        secondary_to_psv_ratio = (
            secondary_peak_value / psv_value
            if psv_value > 0 and not np.isnan(secondary_peak_value)
            else np.nan
        )

        quality_reasons = []

        if time_to_valley_px < min_time_to_valley_px:
            quality_reasons.append("valley_too_early")
        if time_to_valley_px > cycle_length_px * max_time_to_valley_fraction:
            quality_reasons.append("valley_too_late")
        if require_secondary_peak and np.isnan(secondary_peak_local):
            quality_reasons.append("missing_secondary_peak")

        is_complete_beat = len(quality_reasons) == 0
        beat_quality_reason = "ok" if is_complete_beat else ";".join(quality_reasons)

        rows.append(
            {
                "beat_id": i + 1,
                "is_complete_beat": is_complete_beat,
                "beat_quality_reason": beat_quality_reason,
                "psv_local": psv_local,
                "psv_global": psv_global,
                "next_psv_local": next_psv_local,
                "next_psv_global": next_psv_global,
                "psv_value_px": psv_value,
                "first_valley_local": first_valley_local,
                "first_valley_global": first_valley_global,
                "first_valley_value_px": first_valley_value,
                "secondary_peak_local": secondary_peak_local,
                "secondary_peak_global": secondary_peak_global,
                "secondary_peak_value_px": secondary_peak_value,
                "cycle_length_px": cycle_length_px,
                "time_to_valley_px": time_to_valley_px,
                "time_to_secondary_peak_px": time_to_secondary_peak_px,
                "valley_to_psv_ratio": valley_to_psv_ratio,
                "secondary_to_psv_ratio": secondary_to_psv_ratio,
                "ri_proxy": ri_proxy,
            }
        )

    return pd.DataFrame(rows)


def summarize_complete_beats(beat_df):
    """
    This function aggregates complete Doppler beats into recording-level features.
    """
    if len(beat_df) == 0:
        raise ValueError("beat_df is empty.")

    complete_beats = beat_df[beat_df["is_complete_beat"]].copy()
    n_total = len(beat_df)
    n_complete = len(complete_beats)

    if n_complete == 0:
        return pd.DataFrame(
            [{"n_total_beats": n_total, "n_complete_beats": 0, "complete_fraction": 0.0}]
        )

    summary = {
        "n_total_beats": n_total,
        "n_complete_beats": n_complete,
        "complete_fraction": n_complete / n_total,
        "mean_psv_px": complete_beats["psv_value_px"].mean(),
        "std_psv_px": complete_beats["psv_value_px"].std(),
        "mean_edv_proxy_px": complete_beats["first_valley_value_px"].mean(),
        "mean_ri_proxy": complete_beats["ri_proxy"].mean(),
        "std_ri_proxy": complete_beats["ri_proxy"].std(),
        "mean_cycle_length_px": complete_beats["cycle_length_px"].mean(),
        "mean_time_to_valley_px": complete_beats["time_to_valley_px"].mean(),
        "mean_time_to_secondary_peak_px": complete_beats["time_to_secondary_peak_px"].mean(),
        "mean_secondary_to_psv_ratio": complete_beats["secondary_to_psv_ratio"].mean(),
    }

    return pd.DataFrame([summary])

In [7]:
#| export calibration
import numpy as np
import pandas as pd


def fit_velocity_calibration_from_points(calibration_points_df):
    """
    This function fits Doppler velocity calibration from manually selected scale points.
    """
    required_columns = ["calibration_group", "velocity_cm_s", "y_global_px"]
    missing_columns = [
        column for column in required_columns
        if column not in calibration_points_df.columns
    ]

    if missing_columns:
        raise ValueError(
            "Calibration points table is missing required columns:\n"
            + "\n".join(missing_columns)
        )

    calibration_rows = []

    for calibration_group, group_df in calibration_points_df.groupby("calibration_group"):
        group_df = group_df.copy()

        if len(group_df) < 2:
            raise ValueError(f"Calibration group '{calibration_group}' has fewer than 2 points.")

        y = group_df["y_global_px"].to_numpy(dtype=float)
        v = group_df["velocity_cm_s"].to_numpy(dtype=float)

        slope_cm_s_per_px, intercept_cm_s = np.polyfit(y, v, deg=1)
        predicted_velocity_cm_s = slope_cm_s_per_px * y + intercept_cm_s
        residuals_cm_s = v - predicted_velocity_cm_s
        cm_s_per_px = abs(float(slope_cm_s_per_px))

        baseline_candidates = group_df[
            np.isclose(group_df["velocity_cm_s"].astype(float), 0.0)
        ]

        if len(baseline_candidates) > 0:
            baseline_y_global_px = float(baseline_candidates["y_global_px"].iloc[0])
        else:
            baseline_y_global_px = float(-intercept_cm_s / slope_cm_s_per_px)

        calibration_rows.append(
            {
                "calibration_group": calibration_group,
                "n_points": len(group_df),
                "slope_cm_s_per_px": float(slope_cm_s_per_px),
                "intercept_cm_s": float(intercept_cm_s),
                "cm_s_per_px": cm_s_per_px,
                "baseline_y_global_px": baseline_y_global_px,
                "baseline_y_roi_px": baseline_y_global_px - 230.0,
                "max_abs_residual_cm_s": float(np.max(np.abs(residuals_cm_s))),
                "mean_abs_residual_cm_s": float(np.mean(np.abs(residuals_cm_s))),
            }
        )

    return pd.DataFrame(calibration_rows)

In [8]:
#| export video
from pathlib import Path
import cv2


def read_video_frame_by_index_v2(video_path, frame_idx):
    """
    This function reads a video frame by frame index and returns frame timing information.
    """
    video_path = Path(video_path)
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        cap.release()
        raise RuntimeError(f"Invalid FPS detected for video: {video_path}")

    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame_bgr = cap.read()
    cap.release()

    if not ok:
        raise RuntimeError(f"Could not read frame_idx={frame_idx} from video: {video_path}")

    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    frame_time_s = int(frame_idx) / fps

    return {
        "frame_rgb": frame_rgb,
        "fps": float(fps),
        "frame_idx": int(frame_idx),
        "frame_time_s": float(frame_time_s),
    }

In [9]:
baseline_helper_names = [
    "extract_envelope_directional",
    "extract_doppler_waveform",
    "analyze_beats",
    "summarize_complete_beats",
    "fit_velocity_calibration_from_points",
    "read_video_frame_by_index_v2",
]

print("Baseline helper availability")
print("-" * 88)

for helper_name in baseline_helper_names:
    helper_obj = globals().get(helper_name)
    status = "OK" if callable(helper_obj) else "MISSING"
    module_target = {
        "extract_envelope_directional": "waveform",
        "extract_doppler_waveform": "waveform",
        "analyze_beats": "waveform",
        "summarize_complete_beats": "waveform",
        "fit_velocity_calibration_from_points": "calibration",
        "read_video_frame_by_index_v2": "video",
    }[helper_name]
    print(f"{helper_name:42} {status:8} planned nbdev module: {module_target}")

if any(not callable(globals().get(name)) for name in baseline_helper_names):
    raise RuntimeError("One or more baseline helper functions were not defined.")

Baseline helper availability
----------------------------------------------------------------------------------------
extract_envelope_directional               OK       planned nbdev module: waveform
extract_doppler_waveform                   OK       planned nbdev module: waveform
analyze_beats                              OK       planned nbdev module: waveform
summarize_complete_beats                   OK       planned nbdev module: waveform
fit_velocity_calibration_from_points       OK       planned nbdev module: calibration
read_video_frame_by_index_v2               OK       planned nbdev module: video


## Baseline frame manifest and single-frame smoke test

### What this tests

This section tests whether the NB04 V2 frame-level CSV can be loaded and whether one AVI frame can be decoded and processed through the baseline image-path helpers.

### Why this matters

The full-frame baseline reproduction decodes many video frames. A one-frame smoke test catches wiring errors before the heavier validation run.


In [10]:
MANUAL_ROI = {
    "x_min": 80,
    "x_max": 640,
    "y_min": 230,
    "y_max": 510,
}

frames = pd.read_csv(paths["nb04_v2_frame_csv"])
frames_ok = frames[frames["image_extraction_ok"] == True].copy()

frames_ok["video_path"] = frames_ok["recording_name"].map(
    lambda recording_name: paths["avi_batch_dir"] / f"{recording_name}.avi"
)

frames_ok["video_exists"] = frames_ok["video_path"].map(lambda path: Path(path).exists())
frames_ok = frames_ok[frames_ok["video_exists"]].reset_index(drop=True)

sample_row = frames_ok.iloc[0]
sample_frame = read_video_frame_by_index_v2(
    video_path=sample_row["video_path"],
    frame_idx=int(sample_row["video_frame_idx"]),
)

sample_result = extract_doppler_waveform(
    frame_rgb=sample_frame["frame_rgb"],
    **MANUAL_ROI,
)

sample_reproduced_velocity = (
    float(np.nanmax(sample_result["velocity_smooth"]))
    * float(sample_row["cm_s_per_px"])
)

sample_delta = sample_reproduced_velocity - float(sample_row["max_velocity_cm_s"])

print("Baseline manifest and one-frame smoke test")
print("-" * 88)
print(f"NB04 V2 frame rows total:           {len(frames)}")
print(f"image_extraction_ok rows:           {int((frames['image_extraction_ok'] == True).sum())}")
print(f"image_extraction_ok with AVI file:  {len(frames_ok)}")
print("-" * 88)
print(f"sample recording:                   {sample_row['recording_name']}")
print(f"sample video_frame_idx:             {int(sample_row['video_frame_idx'])}")
print(f"sample frame_time_s:                {sample_frame['frame_time_s']:.3f}")
print(f"sample direction:                   {sample_result['direction']}")
print(f"sample auto baseline roi px:        {sample_result['baseline']}")
print(f"sample stored max_velocity_cm_s:    {float(sample_row['max_velocity_cm_s']):.8f}")
print(f"sample reproduced max_velocity_cm_s:{sample_reproduced_velocity:.8f}")
print(f"sample delta cm_s:                  {sample_delta:.12f}")

if abs(sample_delta) >= 0.01:
    raise ValueError(
        f"Single-frame smoke test did not reproduce stored velocity closely enough: delta={sample_delta}"
    )

Baseline manifest and one-frame smoke test
----------------------------------------------------------------------------------------
NB04 V2 frame rows total:           406
image_extraction_ok rows:           406
image_extraction_ok with AVI file:  406
----------------------------------------------------------------------------------------
sample recording:                   candidate_test_01_brachial
sample video_frame_idx:             50
sample frame_time_s:                1.667
sample direction:                   below
sample auto baseline roi px:        146
sample stored max_velocity_cm_s:    23.76035920
sample reproduced max_velocity_cm_s:23.76035920
sample delta cm_s:                  -0.000000000000


## Full-frame baseline reproduction

### What this tests

This section tests whether the copied baseline image-path helper functions reproduce the stored NB04 V2 `max_velocity_cm_s` value for every `image_extraction_ok` frame.

### Why this matters

Native-assisted calibration and robustness experiments need a trusted comparison anchor. Full-frame reproduction confirms that later candidate deltas are measured against the current AVI image-pipeline behavior rather than against a changed implementation.

In [13]:
NB13_OUT_DIR = paths["feature_exports_dir"] / "nb13_validation"
NB13_OUT_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_FRAME_RESULTS = {}
baseline_rows = []

for row_index, frame_row in frames_ok.iterrows():
    recording_name = frame_row["recording_name"]
    video_frame_idx = int(frame_row["video_frame_idx"])
    result_key = (recording_name, video_frame_idx)

    frame = read_video_frame_by_index_v2(
        video_path=frame_row["video_path"],
        frame_idx=video_frame_idx,
    )

    extraction = extract_doppler_waveform(
        frame_rgb=frame["frame_rgb"],
        **MANUAL_ROI,
    )

    reproduced_velocity_cm_s = (
        float(np.nanmax(extraction["velocity_smooth"]))
        * float(frame_row["cm_s_per_px"])
    )

    stored_velocity_cm_s = float(frame_row["max_velocity_cm_s"])
    delta_cm_s = reproduced_velocity_cm_s - stored_velocity_cm_s

    BASELINE_FRAME_RESULTS[result_key] = {
        "velocity_smooth": extraction["velocity_smooth"],
        "baseline": extraction["baseline"],
        "direction": extraction["direction"],
        "peaks": extraction["peaks"],
        "peaks_local": extraction["peaks_local"],
        "peaks_global": extraction["peaks_global"],
        "x_offset": extraction["x_offset"],
    }

    baseline_rows.append(
        {
            "recording_name": recording_name,
            "video_frame_idx": video_frame_idx,
            "stored_max_velocity_cm_s": stored_velocity_cm_s,
            "reproduced_max_velocity_cm_s": reproduced_velocity_cm_s,
            "delta_cm_s": delta_cm_s,
            "abs_delta_cm_s": abs(delta_cm_s),
            "direction": extraction["direction"],
            "auto_baseline_roi_px": extraction["baseline"],
            "n_peaks": int(len(extraction["peaks"])),
        }
    )

baseline_reproduction = pd.DataFrame(baseline_rows)

baseline_summary = (
    baseline_reproduction
    .groupby("recording_name")
    .agg(
        n_frames=("abs_delta_cm_s", "size"),
        median_abs_delta_cm_s=("abs_delta_cm_s", "median"),
        max_abs_delta_cm_s=("abs_delta_cm_s", "max"),
        exact_fraction_0p01=("abs_delta_cm_s", lambda values: float((values < 0.01).mean())),
    )
    .reset_index()
)

baseline_csv_path = NB13_OUT_DIR / "nb13_baseline_reproduction_full.csv"
baseline_reproduction.to_csv(baseline_csv_path, index=False)

print("Full-frame baseline reproduction")
print("-" * 88)
print(f"frames evaluated:                 {len(baseline_reproduction)}")
print(f"median absolute delta cm/s:        {baseline_reproduction['abs_delta_cm_s'].median():.12f}")
print(f"max absolute delta cm/s:           {baseline_reproduction['abs_delta_cm_s'].max():.12f}")
print(f"exact fraction < 0.01 cm/s:        {(baseline_reproduction['abs_delta_cm_s'] < 0.01).mean():.3f}")
print(f"cached baseline frame results:     {len(BASELINE_FRAME_RESULTS)}")
print(f"saved validation CSV:              {baseline_csv_path}")
print("-" * 88)

display(baseline_summary)

if len(baseline_reproduction) != 406:
    raise ValueError(f"Expected 406 baseline rows, found {len(baseline_reproduction)}")

if baseline_reproduction["abs_delta_cm_s"].median() >= 0.01:
    raise ValueError("Baseline reproduction median absolute delta is larger than expected.")

if baseline_reproduction["abs_delta_cm_s"].max() >= 0.01:
    raise ValueError("Baseline reproduction max absolute delta is larger than expected.")

if (baseline_reproduction["abs_delta_cm_s"] < 0.01).mean() != 1.0:
    raise ValueError("Not all baseline rows reproduced within <0.01 cm/s.")

Full-frame baseline reproduction
----------------------------------------------------------------------------------------
frames evaluated:                 406
median absolute delta cm/s:        0.000000000000
max absolute delta cm/s:           0.000000000000
exact fraction < 0.01 cm/s:        1.000
cached baseline frame results:     406
saved validation CSV:              D:\code\DopplerLab\feature_exports\nb13_validation\nb13_baseline_reproduction_full.csv
----------------------------------------------------------------------------------------


,recording_name,n_frames,median_abs_delta_cm_s,max_abs_delta_cm_s,exact_fraction_0p01
0,candidate_test_01_brachial,34,3.552714e-15,7.105427e-15,1.0
1,candidate_test_02_brachial,54,7.105427e-15,1.065814e-14,1.0
2,candidate_test_02_neck,49,5.329071e-15,1.065814e-14,1.0
3,candidate_test_03_brachial,43,7.105427e-15,1.065814e-14,1.0
4,candidate_test_04_brachial_1,55,7.105427e-15,1.065814e-14,1.0
5,candidate_test_04_brachial_2,54,7.105427e-15,1.065814e-14,1.0
6,candidate_test_04_neck,26,4.440892e-15,7.105427e-15,1.0
7,candidate_test_05_brachial,45,7.105427e-15,1.421085e-14,1.0
8,candidate_test_08_brachial,46,7.105427e-15,1.065814e-14,1.0


## Native DcmRegionPara metadata extraction

### What this tests

This section tests whether each mapped native recording contains a PW `DcmRegionPara` entry with ROI bounds, zero-velocity baseline row, velocity scale, and time scale metadata.

### Why this matters

Native metadata may provide a reproducible, per-recording source of ROI and calibration information. This must be parsed and compared against the manual AVI image-path calibration before any candidate velocity deltas are tested.


In [14]:
#| export native_metadata
import re
from pathlib import Path


def parse_dcm_region_pw(native_dir):
    """
    This function reads the PW DcmRegionPara entry from a native Mindray export folder.

    The returned values are metadata-derived ROI and calibration inputs.
    They are not clinical measurements by themselves.
    """
    native_dir = Path(native_dir)
    dcm_region_path = native_dir / "DcmRegionPara.txt"

    if not dcm_region_path.exists():
        raise FileNotFoundError(f"DcmRegionPara.txt was not found: {dcm_region_path}")

    text = dcm_region_path.read_text(errors="replace")
    blocks = re.split(r"DATA_TREE_BEGIN=DcmRegion\d+", text)

    for block in blocks:
        key_values = dict(re.findall(r"(\w+)=([-\d.]+)", block))

        if key_values.get("DataType") == "3" and key_values.get("SpatialFormat") == "3":
            x0 = int(key_values["X0"])
            x1 = int(key_values["X1"])
            y0 = int(key_values["Y0"])
            y1 = int(key_values["Y1"])
            vir_y = int(key_values["VirY"])

            return {
                "x0": x0,
                "x1": x1,
                "y0": y0,
                "y1": y1,
                "vir_y": vir_y,
                "baseline_y_global": y0 + vir_y,
                "baseline_y_roi": vir_y,
                "cm_s_per_px": abs(float(key_values["PhyDeltaY"])),
                "s_per_px": abs(float(key_values["PhyDeltaX"])),
                "data_type": int(key_values["DataType"]),
                "spatial_format": int(key_values["SpatialFormat"]),
            }

    raise ValueError(f"No PW DcmRegion entry found in {dcm_region_path}")

In [15]:
NATIVE_METADATA_BY_RECORDING_ID = {}

metadata_rows = []

manual_roi_text = (
    f"{MANUAL_ROI['x_min']},{MANUAL_ROI['x_max']},"
    f"{MANUAL_ROI['y_min']},{MANUAL_ROI['y_max']}"
)

manual_baseline_global = 375.0
manual_cm_s_per_px = float(frames_ok["cm_s_per_px"].median())

for recording_id, nb04_recording_name in RECORDING_ID_TO_NB04.items():
    native_dir = paths["native_batch_dir"] / recording_id / "native"
    native_metadata = parse_dcm_region_pw(native_dir)
    NATIVE_METADATA_BY_RECORDING_ID[recording_id] = native_metadata

    native_roi_text = (
        f"{native_metadata['x0']},{native_metadata['x1']},"
        f"{native_metadata['y0']},{native_metadata['y1']}"
    )

    metadata_rows.append(
        {
            "recording_id": recording_id,
            "nb04_recording_name": nb04_recording_name,
            "native_roi": native_roi_text,
            "manual_roi": manual_roi_text,
            "native_baseline_global": native_metadata["baseline_y_global"],
            "manual_baseline_global": manual_baseline_global,
            "baseline_delta_px": native_metadata["baseline_y_global"] - manual_baseline_global,
            "native_cm_s_per_px": native_metadata["cm_s_per_px"],
            "manual_cm_s_per_px": manual_cm_s_per_px,
            "scale_delta_pct": 100.0 * (native_metadata["cm_s_per_px"] - manual_cm_s_per_px) / manual_cm_s_per_px,
            "native_s_per_px": native_metadata["s_per_px"],
        }
    )

native_metadata_comparison = pd.DataFrame(metadata_rows).sort_values("recording_id").reset_index(drop=True)

metadata_csv_path = NB13_OUT_DIR / "nb13_native_metadata_calibration.csv"
native_metadata_comparison.to_csv(metadata_csv_path, index=False)

print("Native DcmRegionPara metadata extraction")
print("-" * 88)
print(f"mapped native recordings parsed:   {len(native_metadata_comparison)}")
print(f"unique native ROI values:          {native_metadata_comparison['native_roi'].nunique()}")
print(f"unique native baseline values:     {native_metadata_comparison['native_baseline_global'].nunique()}")
print(f"unique native cm/s/px values:      {native_metadata_comparison['native_cm_s_per_px'].nunique()}")
print(f"manual ROI:                        {manual_roi_text}")
print(f"manual baseline global px:         {manual_baseline_global:.1f}")
print(f"manual cm/s/px median:             {manual_cm_s_per_px:.6f}")
print(f"saved metadata CSV:                {metadata_csv_path}")
print("-" * 88)

display(
    native_metadata_comparison[
        [
            "recording_id",
            "nb04_recording_name",
            "native_roi",
            "manual_roi",
            "native_baseline_global",
            "manual_baseline_global",
            "baseline_delta_px",
            "native_cm_s_per_px",
            "manual_cm_s_per_px",
            "scale_delta_pct",
            "native_s_per_px",
        ]
    ]
)

if len(native_metadata_comparison) != 10:
    raise ValueError(f"Expected metadata for 10 native recordings, found {len(native_metadata_comparison)}")

if native_metadata_comparison["native_cm_s_per_px"].isna().any():
    raise ValueError("At least one native cm/s/px value is missing.")

if native_metadata_comparison["native_s_per_px"].isna().any():
    raise ValueError("At least one native s/px value is missing.")

Native DcmRegionPara metadata extraction
----------------------------------------------------------------------------------------
mapped native recordings parsed:   10
unique native ROI values:          1
unique native baseline values:     1
unique native cm/s/px values:      1
manual ROI:                        80,640,230,510
manual baseline global px:         375.0
manual cm/s/px median:             0.175434
saved metadata CSV:                D:\code\DopplerLab\feature_exports\nb13_validation\nb13_native_metadata_calibration.csv
----------------------------------------------------------------------------------------


,recording_id,nb04_recording_name,native_roi,manual_roi,native_baseline_global,manual_baseline_global,baseline_delta_px,native_cm_s_per_px,manual_cm_s_per_px,scale_delta_pct,native_s_per_px
0,202606130411060002SMP,candidate_test_02_brachial,"79,659,245,508","80,640,230,510",376,375.0,1.0,0.176946,0.175434,0.861673,0.006
1,202606130413540003SMP,candidate_test_03_brachial,"79,659,245,508","80,640,230,510",376,375.0,1.0,0.176946,0.175434,0.861673,0.006
2,202606130417060004SMP,candidate_test_06_brachial,"79,659,245,508","80,640,230,510",376,375.0,1.0,0.176946,0.175434,0.861673,0.006
3,202606130420260005SMP,candidate_test_08_brachial,"79,659,245,508","80,640,230,510",376,375.0,1.0,0.176946,0.175434,0.861673,0.006
4,202606130422440006SMP,candidate_test_02_neck,"79,659,245,508","80,640,230,510",376,375.0,1.0,0.176946,0.175434,0.861673,0.006
5,202606130426500007SMP,candidate_test_05_brachial,"79,659,245,508","80,640,230,510",376,375.0,1.0,0.176946,0.175434,0.861673,0.006
6,202606130430380008SMP,candidate_test_07_brachial,"79,659,245,508","80,640,230,510",376,375.0,1.0,0.176946,0.175434,0.861673,0.006
7,202606130433280009SMP,candidate_test_04_brachial_2,"79,659,245,508","80,640,230,510",376,375.0,1.0,0.176946,0.175434,0.861673,0.006
8,202606130434130010SMP,candidate_test_04_brachial_1,"79,659,245,508","80,640,230,510",376,375.0,1.0,0.176946,0.175434,0.861673,0.006
9,202606130437480012SMP,candidate_test_01_brachial,"79,659,245,508","80,640,230,510",376,375.0,1.0,0.176946,0.175434,0.861673,0.006


## Native PW page structure and counter integrity

### What this tests

This section tests whether each mapped native `PW_CinePartition0.bin` file can be read as fixed-size pages and whether the counter-like bytes near the end of each page behave consistently.

### Why this matters

The native timing/QC sidecar depends on a stable page interpretation. A reproducible page count, duration estimate, and 20 Hz counter pattern are prerequisite checks before deriving `hp_activity`.


In [16]:
#| export native_qc
import numpy as np
import pandas as pd
from pathlib import Path


PAGE_SIZE_BYTES = 1296
PAGE_RATE_HZ = 500.0
COUNTER_BYTE_OFFSET = 1248


def load_native_pw_pages(pw_path, page_size_bytes=PAGE_SIZE_BYTES):
    """
    This function reads PW_CinePartition0.bin as fixed-size native pages.

    Any trailing incomplete bytes are ignored. The returned pages are raw uint8
    bytes and are not interpreted as a decoded spectrogram or velocity signal.
    """
    pw_path = Path(pw_path)

    if not pw_path.exists():
        raise FileNotFoundError(f"Native PW file was not found: {pw_path}")

    raw_bytes = np.fromfile(pw_path, dtype=np.uint8)
    n_pages = raw_bytes.size // page_size_bytes
    trailing_bytes = raw_bytes.size - (n_pages * page_size_bytes)

    if n_pages == 0:
        raise ValueError(f"No complete native PW pages found in {pw_path}")

    pages = raw_bytes[: n_pages * page_size_bytes].reshape(n_pages, page_size_bytes)

    return {
        "pages": pages,
        "n_pages": int(n_pages),
        "file_size_bytes": int(raw_bytes.size),
        "trailing_bytes": int(trailing_bytes),
    }


def summarize_native_counter(
    pages,
    counter_byte_offset=COUNTER_BYTE_OFFSET,
    page_rate_hz=PAGE_RATE_HZ,
):
    """
    This function summarizes the counter-like uint16 field in native PW pages.
    """
    if pages.ndim != 2:
        raise ValueError("pages must be a 2D uint8 array.")

    if pages.shape[1] <= counter_byte_offset + 1:
        raise ValueError("counter byte offset is outside the page width.")

    counter = (
        pages[:, counter_byte_offset].astype(np.uint16)
        | (pages[:, counter_byte_offset + 1].astype(np.uint16) << 8)
    ).astype(np.int64)

    counter_diff = np.diff(counter)
    positive_increment_locations = np.where(counter_diff > 0)[0]

    if positive_increment_locations.size > 2:
        counter_group_pages = float(np.median(np.diff(positive_increment_locations)))
    else:
        counter_group_pages = np.nan

    if np.isfinite(counter_group_pages) and counter_group_pages > 0:
        counter_group_hz = float(page_rate_hz / counter_group_pages)
    else:
        counter_group_hz = np.nan

    return {
        "counter_min": int(counter.min()),
        "counter_max": int(counter.max()),
        "counter_unique": int(np.unique(counter).size),
        "counter_group_pages": counter_group_pages,
        "counter_group_hz": counter_group_hz,
        "counter_resets": int((counter_diff < -100).sum()),
        "counter_monotone_fraction": float((counter_diff >= 0).mean()),
    }

In [17]:
counter_rows = []

for recording_id, nb04_recording_name in RECORDING_ID_TO_NB04.items():
    pw_path = paths["native_batch_dir"] / recording_id / "native" / "PW_CinePartition0.bin"

    page_info = load_native_pw_pages(pw_path)
    counter_summary = summarize_native_counter(page_info["pages"])

    counter_rows.append(
        {
            "recording_id": recording_id,
            "nb04_recording_name": nb04_recording_name,
            "file_size_bytes": page_info["file_size_bytes"],
            "n_pages": page_info["n_pages"],
            "trailing_bytes": page_info["trailing_bytes"],
            "native_duration_s": page_info["n_pages"] / PAGE_RATE_HZ,
            "counter_unique": counter_summary["counter_unique"],
            "counter_group_pages": counter_summary["counter_group_pages"],
            "counter_group_hz": counter_summary["counter_group_hz"],
            "counter_resets": counter_summary["counter_resets"],
            "counter_monotone_fraction": counter_summary["counter_monotone_fraction"],
        }
    )

native_counter_summary = pd.DataFrame(counter_rows).sort_values("recording_id").reset_index(drop=True)

print("Native PW page structure and counter integrity")
print("-" * 88)
print(f"native PW files checked:           {len(native_counter_summary)}")
print(f"page size bytes:                   {PAGE_SIZE_BYTES}")
print(f"page rate Hz assumption:           {PAGE_RATE_HZ:.1f}")
print(f"files with trailing bytes:         {int((native_counter_summary['trailing_bytes'] != 0).sum())}")
print(f"median counter group pages:        {native_counter_summary['counter_group_pages'].median():.1f}")
print(f"median counter group Hz:           {native_counter_summary['counter_group_hz'].median():.1f}")
print(f"total counter resets:              {int(native_counter_summary['counter_resets'].sum())}")
print(f"min counter monotone fraction:     {native_counter_summary['counter_monotone_fraction'].min():.4f}")
print("-" * 88)

display(native_counter_summary)

if len(native_counter_summary) != 10:
    raise ValueError(f"Expected 10 native PW files, found {len(native_counter_summary)}")

if (native_counter_summary["trailing_bytes"] != 0).any():
    raise ValueError("At least one native PW file has trailing bytes after fixed-page parsing.")

if not np.allclose(native_counter_summary["counter_group_pages"], 25.0, equal_nan=False):
    raise ValueError("At least one native PW counter group interval is not 25 pages.")

if native_counter_summary["counter_resets"].sum() != 0:
    raise ValueError("At least one native PW counter reset was detected.")

if (native_counter_summary["counter_monotone_fraction"] < 0.99).any():
    raise ValueError("At least one native PW counter is not mostly monotone.")

Native PW page structure and counter integrity
----------------------------------------------------------------------------------------
native PW files checked:           10
page size bytes:                   1296
page rate Hz assumption:           500.0
files with trailing bytes:         0
median counter group pages:        25.0
median counter group Hz:           20.0
total counter resets:              0
min counter monotone fraction:     1.0000
----------------------------------------------------------------------------------------


,recording_id,nb04_recording_name,file_size_bytes,n_pages,trailing_bytes,native_duration_s,counter_unique,counter_group_pages,counter_group_hz,counter_resets,counter_monotone_fraction
0,202606130411060002SMP,candidate_test_02_brachial,38381040,29615,0,59.230,1180,25.0,20.0,0,1.0
1,202606130413540003SMP,candidate_test_03_brachial,30444336,23491,0,46.982,936,25.0,20.0,0,1.0
2,202606130417060004SMP,candidate_test_06_brachial,27516672,21232,0,42.464,846,25.0,20.0,0,1.0
3,202606130420260005SMP,candidate_test_08_brachial,29502144,22764,0,45.528,907,25.0,20.0,0,1.0
4,202606130422440006SMP,candidate_test_02_neck,33211296,25626,0,51.252,1021,25.0,20.0,0,1.0
5,202606130426500007SMP,candidate_test_05_brachial,23776416,18346,0,36.692,731,25.0,20.0,0,1.0
6,202606130430380008SMP,candidate_test_07_brachial,28753056,22186,0,44.372,884,25.0,20.0,0,1.0
7,202606130433280009SMP,candidate_test_04_brachial_2,20557152,15862,0,31.724,632,25.0,20.0,0,1.0
8,202606130434130010SMP,candidate_test_04_brachial_1,21923136,16916,0,33.832,674,25.0,20.0,0,1.0
9,202606130437480012SMP,candidate_test_01_brachial,29176848,22513,0,45.026,897,25.0,20.0,0,1.0


## Native hp_activity timing/QC sidecar

### What this tests

This section tests whether an experimental `hp_activity` timing/QC sidecar can be reproduced directly from raw `PW_CinePartition0.bin` bytes.

### Why this matters

A native timing/QC sidecar may provide additive timing support for AVI/audio review, but only if it is reproducible from raw native files and remains clearly separated from velocity-envelope or clinical-measurement claims.


In [18]:
#| export native_qc
from scipy import signal as sg
from scipy.ndimage import uniform_filter1d


HP_ENV_RATE_HZ = 100.0


def load_native_pw_record_field_b(pw_path):
    """
    This function loads the dynamic record field used for experimental native
    timing/QC sidecar extraction.

    The returned array is an arbitrary-unit compact native page feature. It is
    not interpreted as a decoded spectrogram or velocity envelope.
    """
    page_info = load_native_pw_pages(pw_path)
    pages = page_info["pages"]
    n_pages = page_info["n_pages"]

    records = pages[:, 16:144].reshape(n_pages, 16, 8)

    field_b = (
        np.ascontiguousarray(records[:, :14, 4:8])
        .reshape(-1, 4)
        .view("<f4")
        .reshape(n_pages, 14)
    )

    field_b = np.nan_to_num(field_b.astype(np.float64))

    return field_b, page_info


def compute_hp_activity(field_b, smoothing_window_pages=51):
    """
    This function computes an arbitrary-unit high-pass activity trace from
    compact native PW page records.
    """
    smoothed = uniform_filter1d(
        field_b,
        size=smoothing_window_pages,
        axis=0,
        mode="nearest",
    )

    hp_activity = np.abs(field_b - smoothed).mean(axis=1)

    return hp_activity


def compute_activity_envelope(
    activity,
    page_rate_hz=PAGE_RATE_HZ,
    envelope_rate_hz=HP_ENV_RATE_HZ,
    band_hz=(0.5, 12.0),
):
    """
    This function computes a low-rate envelope from the experimental native
    activity trace for timing/QC estimation.
    """
    activity = np.asarray(activity, dtype=float)

    if activity.ndim != 1:
        raise ValueError("activity must be a 1D array.")

    if not np.isfinite(activity).all():
        raise ValueError("activity contains non-finite values.")

    centered = activity - activity.mean()

    sos = sg.butter(
        N=4,
        Wn=[band_hz[0] / (page_rate_hz / 2.0), band_hz[1] / (page_rate_hz / 2.0)],
        btype="band",
        output="sos",
    )

    filtered = sg.sosfiltfilt(sos, centered)
    envelope = np.abs(sg.hilbert(filtered))

    downsample_factor = int(round(page_rate_hz / envelope_rate_hz))
    envelope_low_rate = envelope[::downsample_factor]

    return envelope_low_rate


def estimate_periodic_hr_bpm(envelope, envelope_rate_hz=HP_ENV_RATE_HZ, low_bpm=40, high_bpm=200):
    """
    This function estimates a dominant HR-like periodicity from an envelope
    autocorrelation.
    """
    envelope = np.asarray(envelope, dtype=float)

    if envelope.ndim != 1:
        raise ValueError("envelope must be a 1D array.")

    signal_centered = envelope - envelope.mean()

    if signal_centered.std() == 0:
        return {
            "autocorr_peak": 0.0,
            "hr_bpm": np.nan,
        }

    n_samples = len(signal_centered)
    fft_values = np.fft.rfft(signal_centered, n=2 * n_samples)
    autocorr = np.fft.irfft(fft_values * np.conj(fft_values))[:n_samples]
    autocorr = autocorr / autocorr[0]

    min_lag = int(envelope_rate_hz * 60.0 / high_bpm)
    max_lag = min(int(envelope_rate_hz * 60.0 / low_bpm), n_samples - 1)

    if max_lag <= min_lag:
        return {
            "autocorr_peak": np.nan,
            "hr_bpm": np.nan,
        }

    local_peak_index = int(np.argmax(autocorr[min_lag:max_lag]))
    lag_samples = min_lag + local_peak_index
    hr_bpm = 60.0 / (lag_samples / envelope_rate_hz)

    return {
        "autocorr_peak": float(autocorr[lag_samples]),
        "hr_bpm": float(hr_bpm),
    }


def summarize_native_hp_activity_sidecar(pw_path):
    """
    This function derives the experimental native hp_activity timing/QC sidecar
    from one raw PW_CinePartition0.bin file.
    """
    field_b, page_info = load_native_pw_record_field_b(pw_path)
    hp_activity = compute_hp_activity(field_b)
    envelope = compute_activity_envelope(hp_activity)
    periodicity = estimate_periodic_hr_bpm(envelope)

    window_length = len(envelope) // 5
    window_hr_values = []

    if window_length > HP_ENV_RATE_HZ * 4:
        for window_index in range(5):
            start = window_index * window_length
            stop = (window_index + 1) * window_length
            window_result = estimate_periodic_hr_bpm(envelope[start:stop])
            window_hr_values.append(window_result["hr_bpm"])

    window_hr_values = np.asarray(window_hr_values, dtype=float)
    finite_window_hr = window_hr_values[np.isfinite(window_hr_values)]

    if len(finite_window_hr) > 2:
        window_hr_iqr_bpm = float(
            np.percentile(finite_window_hr, 75)
            - np.percentile(finite_window_hr, 25)
        )
    else:
        window_hr_iqr_bpm = np.nan

    def subset_hr(column_indices):
        subset_activity = compute_hp_activity(field_b[:, column_indices])
        subset_envelope = compute_activity_envelope(subset_activity)
        return estimate_periodic_hr_bpm(subset_envelope)["hr_bpm"]

    first_half_hr = subset_hr(list(range(0, 7)))
    second_half_hr = subset_hr(list(range(7, 14)))
    even_hr = subset_hr(list(range(0, 14, 2)))
    odd_hr = subset_hr(list(range(1, 14, 2)))

    subset_disagreement_bpm = float(
        np.mean(
            [
                abs(first_half_hr - second_half_hr),
                abs(even_hr - odd_hr),
            ]
        )
    )

    native_specific_reproduced = bool(
        window_hr_iqr_bpm < 8.0
        and subset_disagreement_bpm < 6.0
    )

    return {
        "n_pages": page_info["n_pages"],
        "native_duration_s": page_info["n_pages"] / PAGE_RATE_HZ,
        "hp_activity_finite": bool(np.isfinite(hp_activity).all()),
        "hp_activity_min": float(np.min(hp_activity)),
        "hp_activity_max": float(np.max(hp_activity)),
        "native_hr_bpm": round(periodicity["hr_bpm"], 1),
        "autocorr_peak": round(periodicity["autocorr_peak"], 3),
        "window_hr_iqr_bpm": round(window_hr_iqr_bpm, 1) if np.isfinite(window_hr_iqr_bpm) else np.nan,
        "subset_disagreement_bpm": round(subset_disagreement_bpm, 1),
        "native_specific_reproduced": native_specific_reproduced,
    }

In [19]:
PROJECT_NATIVE_QC_CLASS = {
    "202606130411060002SMP": "native_specific",
    "202606130413540003SMP": "native_specific",
    "202606130417060004SMP": "native_specific",
    "202606130420260005SMP": "native_specific",
    "202606130422440006SMP": "native_specific",
    "202606130437480012SMP": "native_specific",
    "202606130433280009SMP": "audio_only",
    "202606130434130010SMP": "audio_only",
    "202606130426500007SMP": "reject",
    "202606130430380008SMP": "reject",
}

sidecar_rows = []

for recording_id, nb04_recording_name in RECORDING_ID_TO_NB04.items():
    pw_path = paths["native_batch_dir"] / recording_id / "native" / "PW_CinePartition0.bin"
    sidecar = summarize_native_hp_activity_sidecar(pw_path)

    reference_class = PROJECT_NATIVE_QC_CLASS.get(recording_id, "unclassified")
    expected_specific = reference_class == "native_specific"

    sidecar_rows.append(
        {
            "recording_id": recording_id,
            "nb04_recording_name": nb04_recording_name,
            "project_native_qc_class": reference_class,
            "native_hr_bpm": sidecar["native_hr_bpm"],
            "autocorr_peak": sidecar["autocorr_peak"],
            "window_hr_iqr_bpm": sidecar["window_hr_iqr_bpm"],
            "subset_disagreement_bpm": sidecar["subset_disagreement_bpm"],
            "native_specific_reproduced": sidecar["native_specific_reproduced"],
            "expected_native_specific": expected_specific,
            "class_agrees": sidecar["native_specific_reproduced"] == expected_specific,
            "hp_activity_finite": sidecar["hp_activity_finite"],
            "hp_activity_min": sidecar["hp_activity_min"],
            "hp_activity_max": sidecar["hp_activity_max"],
            "native_duration_s": sidecar["native_duration_s"],
        }
    )

native_hp_sidecar = pd.DataFrame(sidecar_rows).sort_values("recording_id").reset_index(drop=True)

sidecar_csv_path = NB13_OUT_DIR / "nb13_native_qc_sidecar.csv"
native_hp_sidecar.to_csv(sidecar_csv_path, index=False)

print("Native hp_activity timing/QC sidecar")
print("-" * 88)
print(f"native sidecars reproduced:         {len(native_hp_sidecar)}")
print(f"hp_activity finite for all:         {bool(native_hp_sidecar['hp_activity_finite'].all())}")
print(f"class agreement count:              {int(native_hp_sidecar['class_agrees'].sum())}/{len(native_hp_sidecar)}")
print(f"native_specific reproduced count:   {int(native_hp_sidecar['native_specific_reproduced'].sum())}")
print(f"saved sidecar CSV:                  {sidecar_csv_path}")
print("-" * 88)

display(
    native_hp_sidecar[
        [
            "recording_id",
            "nb04_recording_name",
            "project_native_qc_class",
            "native_hr_bpm",
            "autocorr_peak",
            "window_hr_iqr_bpm",
            "subset_disagreement_bpm",
            "native_specific_reproduced",
            "expected_native_specific",
            "class_agrees",
            "hp_activity_finite",
            "native_duration_s",
        ]
    ]
)

if len(native_hp_sidecar) != 10:
    raise ValueError(f"Expected 10 native sidecar rows, found {len(native_hp_sidecar)}")

if not native_hp_sidecar["hp_activity_finite"].all():
    raise ValueError("At least one hp_activity trace contained non-finite values.")

if native_hp_sidecar["class_agrees"].sum() != 10:
    raise ValueError("Native-specific reproduction did not match the project reference labels for all recordings.")

Native hp_activity timing/QC sidecar
----------------------------------------------------------------------------------------
native sidecars reproduced:         10
hp_activity finite for all:         True
class agreement count:              10/10
native_specific reproduced count:   6
saved sidecar CSV:                  D:\code\DopplerLab\feature_exports\nb13_validation\nb13_native_qc_sidecar.csv
----------------------------------------------------------------------------------------


,recording_id,nb04_recording_name,project_native_qc_class,native_hr_bpm,autocorr_peak,window_hr_iqr_bpm,subset_disagreement_bpm,native_specific_reproduced,expected_native_specific,class_agrees,hp_activity_finite,native_duration_s
0,202606130411060002SMP,candidate_test_02_brachial,native_specific,62.5,0.488,4.2,0.3,True,True,True,True,59.230
1,202606130413540003SMP,candidate_test_03_brachial,native_specific,54.1,0.681,1.5,0.5,True,True,True,True,46.982
2,202606130417060004SMP,candidate_test_06_brachial,native_specific,113.2,0.723,0.0,0.0,True,True,True,True,42.464
3,202606130420260005SMP,candidate_test_08_brachial,native_specific,64.5,0.342,4.9,1.7,True,True,True,True,45.528
4,202606130422440006SMP,candidate_test_02_neck,native_specific,58.3,0.577,1.7,0.6,True,True,True,True,51.252
5,202606130426500007SMP,candidate_test_05_brachial,reject,64.5,0.145,73.7,4.3,False,False,True,True,36.692
6,202606130430380008SMP,candidate_test_07_brachial,reject,200.0,0.445,42.1,0.0,False,False,True,True,44.372
7,202606130433280009SMP,candidate_test_04_brachial_2,audio_only,109.1,0.457,54.5,2.0,False,False,True,True,31.724
8,202606130434130010SMP,candidate_test_04_brachial_1,audio_only,107.1,0.353,1.9,29.3,False,False,True,True,33.832
9,202606130437480012SMP,candidate_test_01_brachial,native_specific,65.2,0.474,7.8,0.4,True,True,True,True,45.026


## Native scale-only calibration delta

### What this tests

This section tests the candidate effect of substituting the native `DcmRegionPara` velocity scale while keeping the manual ROI, auto baseline, and cached baseline waveform unchanged.

### Why this matters

Native scale-only calibration isolates one metadata-derived change. If the effect is small and uniform, it can be considered separately from ROI-edge effects and baseline-override effects.


In [20]:
#| export calibration
import numpy as np


def derive_max_velocity_cm_s_from_waveform(
    waveform_result,
    cm_s_per_px,
    roi_y_min=230,
    native_baseline_global=None,
):
    """
    This function derives a max-velocity value from a cached waveform result.

    If native_baseline_global is provided, the function applies a direction-aware
    first-order baseline re-reference. Without that argument, it reproduces the
    existing auto-baseline behavior.
    """
    if waveform_result is None:
        return {
            "ok": False,
            "max_velocity_px": np.nan,
            "max_velocity_cm_s": np.nan,
            "baseline_shift_px": np.nan,
        }

    velocity_smooth = np.asarray(waveform_result["velocity_smooth"], dtype=float)

    if len(velocity_smooth) == 0:
        return {
            "ok": False,
            "max_velocity_px": np.nan,
            "max_velocity_cm_s": np.nan,
            "baseline_shift_px": np.nan,
        }

    max_velocity_px = float(np.nanmax(velocity_smooth))
    baseline_shift_px = 0.0

    if native_baseline_global is not None and np.isfinite(max_velocity_px):
        auto_baseline_roi = float(waveform_result["baseline"])
        native_baseline_roi = float(native_baseline_global) - float(roi_y_min)
        direction = waveform_result.get("direction", "unknown")

        if direction == "above":
            baseline_shift_px = native_baseline_roi - auto_baseline_roi
        elif direction == "below":
            baseline_shift_px = auto_baseline_roi - native_baseline_roi
        else:
            baseline_shift_px = 0.0

        max_velocity_px = max_velocity_px + baseline_shift_px

    return {
        "ok": True,
        "max_velocity_px": max_velocity_px,
        "max_velocity_cm_s": max_velocity_px * float(cm_s_per_px),
        "baseline_shift_px": baseline_shift_px,
    }

In [21]:
mapped_frames = frames_ok[frames_ok["recording_name"].isin(NB04_TO_RECORDING_ID)].copy()
mapped_frames = mapped_frames.reset_index(drop=True)

scale_only_rows = []

for _, frame_row in mapped_frames.iterrows():
    recording_name = frame_row["recording_name"]
    video_frame_idx = int(frame_row["video_frame_idx"])
    recording_id = NB04_TO_RECORDING_ID[recording_name]
    native_metadata = NATIVE_METADATA_BY_RECORDING_ID[recording_id]

    result_key = (recording_name, video_frame_idx)
    cached_waveform = BASELINE_FRAME_RESULTS.get(result_key)

    baseline_metric = derive_max_velocity_cm_s_from_waveform(
        waveform_result=cached_waveform,
        cm_s_per_px=float(frame_row["cm_s_per_px"]),
        roi_y_min=MANUAL_ROI["y_min"],
    )

    candidate_metric = derive_max_velocity_cm_s_from_waveform(
        waveform_result=cached_waveform,
        cm_s_per_px=float(native_metadata["cm_s_per_px"]),
        roi_y_min=MANUAL_ROI["y_min"],
    )

    delta_cm_s = candidate_metric["max_velocity_cm_s"] - baseline_metric["max_velocity_cm_s"]

    scale_only_rows.append(
        {
            "recording_name": recording_name,
            "recording_id": recording_id,
            "video_frame_idx": video_frame_idx,
            "native_class": PROJECT_NATIVE_QC_CLASS.get(recording_id, "unclassified"),
            "baseline_velocity_cm_s": baseline_metric["max_velocity_cm_s"],
            "native_scale_velocity_cm_s": candidate_metric["max_velocity_cm_s"],
            "dvel_cm_s": delta_cm_s,
            "abs_dvel_cm_s": abs(delta_cm_s),
            "manual_cm_s_per_px": float(frame_row["cm_s_per_px"]),
            "native_cm_s_per_px": float(native_metadata["cm_s_per_px"]),
            "scale_delta_pct": 100.0 * (
                float(native_metadata["cm_s_per_px"]) - float(frame_row["cm_s_per_px"])
            ) / float(frame_row["cm_s_per_px"]),
        }
    )

native_scale_only_delta = pd.DataFrame(scale_only_rows)

scale_csv_path = NB13_OUT_DIR / "nb13_delta_native_scale_only.csv"
native_scale_only_delta.to_csv(scale_csv_path, index=False)

scale_summary_by_recording = (
    native_scale_only_delta
    .groupby(["recording_name", "recording_id"])
    .agg(
        n_frames=("dvel_cm_s", "size"),
        median_dvel_cm_s=("dvel_cm_s", "median"),
        median_abs_dvel_cm_s=("abs_dvel_cm_s", "median"),
        max_abs_dvel_cm_s=("abs_dvel_cm_s", "max"),
        median_scale_delta_pct=("scale_delta_pct", "median"),
    )
    .reset_index()
)

print("Native scale-only calibration delta")
print("-" * 88)
print(f"mapped frames evaluated:           {len(native_scale_only_delta)}")
print(f"median dvel cm/s:                  {native_scale_only_delta['dvel_cm_s'].median():.6f}")
print(f"median absolute dvel cm/s:         {native_scale_only_delta['abs_dvel_cm_s'].median():.6f}")
print(f"max absolute dvel cm/s:            {native_scale_only_delta['abs_dvel_cm_s'].max():.6f}")
print(f"frames changed > 0.05 cm/s:        {int((native_scale_only_delta['abs_dvel_cm_s'] > 0.05).sum())}")
print(f"saved delta CSV:                   {scale_csv_path}")
print("-" * 88)

display(scale_summary_by_recording)

if len(native_scale_only_delta) != 380:
    raise ValueError(f"Expected 380 mapped frames, found {len(native_scale_only_delta)}")

if native_scale_only_delta["dvel_cm_s"].isna().any():
    raise ValueError("At least one native scale-only delta is missing.")

Native scale-only calibration delta
----------------------------------------------------------------------------------------
mapped frames evaluated:           380
median dvel cm/s:                  0.165191
median absolute dvel cm/s:         0.165191
max absolute dvel cm/s:            0.219169
frames changed > 0.05 cm/s:        380
saved delta CSV:                   D:\code\DopplerLab\feature_exports\nb13_validation\nb13_delta_native_scale_only.csv
----------------------------------------------------------------------------------------


,recording_name,recording_id,n_frames,median_dvel_cm_s,median_abs_dvel_cm_s,max_abs_dvel_cm_s,median_scale_delta_pct
0,candidate_test_01_brachial,202606130437480012SMP,34,0.114233,0.114233,0.204737,0.861673
1,candidate_test_02_brachial,202606130411060002SMP,54,0.181883,0.181883,0.210975,0.861673
2,candidate_test_02_neck,202606130422440006SMP,49,0.139185,0.139185,0.164191,0.861673
3,candidate_test_03_brachial,202606130413540003SMP,43,0.167118,0.167118,0.182598,0.861673
4,candidate_test_04_brachial_1,202606130434130010SMP,55,0.166898,0.166898,0.199440,0.861673
5,candidate_test_04_brachial_2,202606130433280009SMP,54,0.164826,0.164826,0.198548,0.861673
6,candidate_test_05_brachial,202606130426500007SMP,45,0.213180,0.213180,0.219169,0.861673
7,candidate_test_08_brachial,202606130420260005SMP,46,0.170498,0.170498,0.205882,0.861673


## Native ROI-only calibration delta

### What this tests

This section tests the candidate effect of substituting the native `DcmRegionPara` ROI while keeping the manual velocity scale and auto-baseline behavior unchanged.

### Why this matters

Native ROI-only calibration isolates crop and ROI-edge effects. This differs from scale-only because changing the ROI can add or remove pixels from the extracted waveform.


In [22]:
def native_roi_from_metadata(native_metadata):
    """
    This function converts native DcmRegionPara ROI metadata into the
    argument names expected by extract_doppler_waveform.
    """
    return {
        "x_min": int(native_metadata["x0"]),
        "x_max": int(native_metadata["x1"]),
        "y_min": int(native_metadata["y0"]),
        "y_max": int(native_metadata["y1"]),
    }


def extract_waveform_for_frame_row(frame_row, roi):
    """
    This function reads one AVI frame from a frame-level row and extracts
    the Doppler waveform using the supplied ROI.
    """
    frame = read_video_frame_by_index_v2(
        video_path=frame_row["video_path"],
        frame_idx=int(frame_row["video_frame_idx"]),
    )

    return extract_doppler_waveform(
        frame_rgb=frame["frame_rgb"],
        **roi,
    )

In [23]:
roi_only_rows = []

for _, frame_row in mapped_frames.iterrows():
    recording_name = frame_row["recording_name"]
    video_frame_idx = int(frame_row["video_frame_idx"])
    recording_id = NB04_TO_RECORDING_ID[recording_name]
    native_metadata = NATIVE_METADATA_BY_RECORDING_ID[recording_id]

    result_key = (recording_name, video_frame_idx)
    cached_baseline_waveform = BASELINE_FRAME_RESULTS.get(result_key)

    baseline_metric = derive_max_velocity_cm_s_from_waveform(
        waveform_result=cached_baseline_waveform,
        cm_s_per_px=float(frame_row["cm_s_per_px"]),
        roi_y_min=MANUAL_ROI["y_min"],
    )

    native_roi = native_roi_from_metadata(native_metadata)
    native_roi_waveform = extract_waveform_for_frame_row(frame_row, native_roi)

    candidate_metric = derive_max_velocity_cm_s_from_waveform(
        waveform_result=native_roi_waveform,
        cm_s_per_px=float(frame_row["cm_s_per_px"]),
        roi_y_min=native_roi["y_min"],
    )

    delta_cm_s = candidate_metric["max_velocity_cm_s"] - baseline_metric["max_velocity_cm_s"]

    roi_only_rows.append(
        {
            "recording_name": recording_name,
            "recording_id": recording_id,
            "video_frame_idx": video_frame_idx,
            "native_class": PROJECT_NATIVE_QC_CLASS.get(recording_id, "unclassified"),
            "baseline_velocity_cm_s": baseline_metric["max_velocity_cm_s"],
            "native_roi_velocity_cm_s": candidate_metric["max_velocity_cm_s"],
            "dvel_cm_s": delta_cm_s,
            "abs_dvel_cm_s": abs(delta_cm_s),
            "manual_roi": f"{MANUAL_ROI['x_min']},{MANUAL_ROI['x_max']},{MANUAL_ROI['y_min']},{MANUAL_ROI['y_max']}",
            "native_roi": f"{native_roi['x_min']},{native_roi['x_max']},{native_roi['y_min']},{native_roi['y_max']}",
            "baseline_direction": cached_baseline_waveform.get("direction") if cached_baseline_waveform else None,
            "native_roi_direction": native_roi_waveform.get("direction"),
            "baseline_auto_baseline_roi_px": cached_baseline_waveform.get("baseline") if cached_baseline_waveform else np.nan,
            "native_roi_auto_baseline_roi_px": native_roi_waveform.get("baseline"),
        }
    )

native_roi_only_delta = pd.DataFrame(roi_only_rows)

roi_csv_path = NB13_OUT_DIR / "nb13_delta_native_roi_only.csv"
native_roi_only_delta.to_csv(roi_csv_path, index=False)

roi_summary_by_recording = (
    native_roi_only_delta
    .groupby(["recording_name", "recording_id"])
    .agg(
        n_frames=("dvel_cm_s", "size"),
        median_dvel_cm_s=("dvel_cm_s", "median"),
        median_abs_dvel_cm_s=("abs_dvel_cm_s", "median"),
        max_abs_dvel_cm_s=("abs_dvel_cm_s", "max"),
        frames_changed_gt_0p05=("abs_dvel_cm_s", lambda values: int((values > 0.05).sum())),
        direction_changes=("native_roi_direction", lambda values: int((values != native_roi_only_delta.loc[values.index, "baseline_direction"]).sum())),
    )
    .reset_index()
)

print("Native ROI-only calibration delta")
print("-" * 88)
print(f"mapped frames evaluated:           {len(native_roi_only_delta)}")
print(f"median dvel cm/s:                  {native_roi_only_delta['dvel_cm_s'].median():.6f}")
print(f"median absolute dvel cm/s:         {native_roi_only_delta['abs_dvel_cm_s'].median():.6f}")
print(f"max absolute dvel cm/s:            {native_roi_only_delta['abs_dvel_cm_s'].max():.6f}")
print(f"frames changed > 0.05 cm/s:        {int((native_roi_only_delta['abs_dvel_cm_s'] > 0.05).sum())}")
print(f"saved delta CSV:                   {roi_csv_path}")
print("-" * 88)

display(roi_summary_by_recording)

if len(native_roi_only_delta) != 380:
    raise ValueError(f"Expected 380 mapped frames, found {len(native_roi_only_delta)}")

if native_roi_only_delta["dvel_cm_s"].isna().any():
    raise ValueError("At least one native ROI-only delta is missing.")

Native ROI-only calibration delta
----------------------------------------------------------------------------------------
mapped frames evaluated:           380
median dvel cm/s:                  0.000000
median absolute dvel cm/s:         0.000000
max absolute dvel cm/s:            3.644831
frames changed > 0.05 cm/s:        35
saved delta CSV:                   D:\code\DopplerLab\feature_exports\nb13_validation\nb13_delta_native_roi_only.csv
----------------------------------------------------------------------------------------


,recording_name,recording_id,n_frames,median_dvel_cm_s,median_abs_dvel_cm_s,max_abs_dvel_cm_s,frames_changed_gt_0p05,direction_changes
0,candidate_test_01_brachial,202606130437480012SMP,34,0.0,0.0,0.172474,1,0
1,candidate_test_02_brachial,202606130411060002SMP,54,0.0,0.0,1.266438,2,0
2,candidate_test_02_neck,202606130422440006SMP,49,0.0,0.0,2.524919,8,2
3,candidate_test_03_brachial,202606130413540003SMP,43,0.0,0.0,0.335791,4,0
4,candidate_test_04_brachial_1,202606130434130010SMP,55,0.0,0.0,0.648195,5,0
5,candidate_test_04_brachial_2,202606130433280009SMP,54,0.0,0.0,3.644831,6,2
6,candidate_test_05_brachial,202606130426500007SMP,45,0.0,0.0,2.709058,5,0
7,candidate_test_08_brachial,202606130420260005SMP,46,0.0,0.0,1.458189,4,0


## Native baseline-override-only calibration delta

### What this tests

This section tests the candidate effect of applying the native `DcmRegionPara` zero-velocity baseline as a direction-aware override while keeping the manual ROI, manual velocity scale, and cached baseline waveform unchanged.

### Why this matters

Baseline override isolates the effect of re-referencing velocity distances to the native metadata baseline. The sign of the shift depends on whether the extracted spectrum is above or below the baseline.


In [24]:
baseline_override_rows = []

for _, frame_row in mapped_frames.iterrows():
    recording_name = frame_row["recording_name"]
    video_frame_idx = int(frame_row["video_frame_idx"])
    recording_id = NB04_TO_RECORDING_ID[recording_name]
    native_metadata = NATIVE_METADATA_BY_RECORDING_ID[recording_id]

    result_key = (recording_name, video_frame_idx)
    cached_waveform = BASELINE_FRAME_RESULTS.get(result_key)

    baseline_metric = derive_max_velocity_cm_s_from_waveform(
        waveform_result=cached_waveform,
        cm_s_per_px=float(frame_row["cm_s_per_px"]),
        roi_y_min=MANUAL_ROI["y_min"],
    )

    candidate_metric = derive_max_velocity_cm_s_from_waveform(
        waveform_result=cached_waveform,
        cm_s_per_px=float(frame_row["cm_s_per_px"]),
        roi_y_min=MANUAL_ROI["y_min"],
        native_baseline_global=float(native_metadata["baseline_y_global"]),
    )

    delta_cm_s = candidate_metric["max_velocity_cm_s"] - baseline_metric["max_velocity_cm_s"]

    auto_baseline_roi = float(cached_waveform["baseline"])
    native_baseline_roi = float(native_metadata["baseline_y_global"]) - float(MANUAL_ROI["y_min"])

    baseline_override_rows.append(
        {
            "recording_name": recording_name,
            "recording_id": recording_id,
            "video_frame_idx": video_frame_idx,
            "native_class": PROJECT_NATIVE_QC_CLASS.get(recording_id, "unclassified"),
            "direction": cached_waveform.get("direction"),
            "auto_baseline_roi_px": auto_baseline_roi,
            "native_baseline_roi_px": native_baseline_roi,
            "auto_minus_native_px": auto_baseline_roi - native_baseline_roi,
            "baseline_shift_px_applied": candidate_metric["baseline_shift_px"],
            "baseline_velocity_cm_s": baseline_metric["max_velocity_cm_s"],
            "native_baseline_velocity_cm_s": candidate_metric["max_velocity_cm_s"],
            "dvel_cm_s": delta_cm_s,
            "abs_dvel_cm_s": abs(delta_cm_s),
        }
    )

native_baseline_override_delta = pd.DataFrame(baseline_override_rows)

baseline_override_csv_path = NB13_OUT_DIR / "nb13_delta_native_baseline_override_only.csv"
native_baseline_override_delta.to_csv(baseline_override_csv_path, index=False)

baseline_override_summary = (
    native_baseline_override_delta
    .groupby(["recording_name", "recording_id", "direction"])
    .agg(
        n_frames=("dvel_cm_s", "size"),
        median_dvel_cm_s=("dvel_cm_s", "median"),
        median_abs_dvel_cm_s=("abs_dvel_cm_s", "median"),
        max_abs_dvel_cm_s=("abs_dvel_cm_s", "max"),
        frames_changed_gt_0p05=("abs_dvel_cm_s", lambda values: int((values > 0.05).sum())),
        min_auto_minus_native_px=("auto_minus_native_px", "min"),
        max_auto_minus_native_px=("auto_minus_native_px", "max"),
        min_shift_px_applied=("baseline_shift_px_applied", "min"),
        max_shift_px_applied=("baseline_shift_px_applied", "max"),
    )
    .reset_index()
)

print("Native baseline-override-only calibration delta")
print("-" * 88)
print(f"mapped frames evaluated:           {len(native_baseline_override_delta)}")
print(f"median dvel cm/s:                  {native_baseline_override_delta['dvel_cm_s'].median():.6f}")
print(f"median absolute dvel cm/s:         {native_baseline_override_delta['abs_dvel_cm_s'].median():.6f}")
print(f"max absolute dvel cm/s:            {native_baseline_override_delta['abs_dvel_cm_s'].max():.6f}")
print(f"frames changed > 0.05 cm/s:        {int((native_baseline_override_delta['abs_dvel_cm_s'] > 0.05).sum())}")
print(f"saved delta CSV:                   {baseline_override_csv_path}")
print("-" * 88)

display(baseline_override_summary)

if len(native_baseline_override_delta) != 380:
    raise ValueError(f"Expected 380 mapped frames, found {len(native_baseline_override_delta)}")

if native_baseline_override_delta["dvel_cm_s"].isna().any():
    raise ValueError("At least one native baseline-override delta is missing.")

Native baseline-override-only calibration delta
----------------------------------------------------------------------------------------
mapped frames evaluated:           380
median dvel cm/s:                  0.000000
median absolute dvel cm/s:         0.000000
max absolute dvel cm/s:            3.859550
frames changed > 0.05 cm/s:        51
saved delta CSV:                   D:\code\DopplerLab\feature_exports\nb13_validation\nb13_delta_native_baseline_override_only.csv
----------------------------------------------------------------------------------------


,recording_name,recording_id,direction,n_frames,median_dvel_cm_s,median_abs_dvel_cm_s,max_abs_dvel_cm_s,frames_changed_gt_0p05,min_auto_minus_native_px,max_auto_minus_native_px,min_shift_px_applied,max_shift_px_applied
0,candidate_test_01_brachial,202606130437480012SMP,below,34,0.000000,0.000000,0.000000,0,0.0,0.0,0.0,0.0
1,candidate_test_02_brachial,202606130411060002SMP,below,54,0.000000,0.000000,0.000000,0,0.0,0.0,0.0,0.0
2,candidate_test_02_neck,202606130422440006SMP,above,47,0.000000,0.000000,2.631511,20,-15.0,0.0,0.0,15.0
3,candidate_test_02_neck,202606130422440006SMP,below,2,-2.543794,2.543794,2.631511,2,-15.0,-14.0,-15.0,-14.0
4,candidate_test_03_brachial,202606130413540003SMP,above,43,0.000000,0.000000,0.000000,0,0.0,0.0,0.0,0.0
5,candidate_test_04_brachial_1,202606130434130010SMP,above,53,0.000000,0.000000,0.000000,0,0.0,0.0,0.0,0.0
6,candidate_test_04_brachial_1,202606130434130010SMP,below,2,-3.508682,3.508682,3.508682,2,-20.0,-20.0,-20.0,-20.0
7,candidate_test_04_brachial_2,202606130433280009SMP,above,22,0.000000,0.000000,0.000000,0,0.0,0.0,0.0,0.0
8,candidate_test_04_brachial_2,202606130433280009SMP,below,32,-2.806946,2.806946,3.859550,27,-22.0,0.0,-22.0,0.0
9,candidate_test_05_brachial,202606130426500007SMP,above,30,0.000000,0.000000,0.000000,0,0.0,0.0,0.0,0.0


## Native ROI plus scale calibration delta

### What this tests

This section tests the candidate effect of using both the native `DcmRegionPara` ROI and native velocity scale while leaving auto-baseline behavior unchanged.

### Why this matters

Native ROI plus scale combines two metadata-derived calibration inputs without applying a baseline override. This separates ROI/scale effects from zero-velocity re-reference effects.



In [25]:
NATIVE_ROI_FRAME_RESULTS = {}

roi_plus_scale_rows = []

for _, frame_row in mapped_frames.iterrows():
    recording_name = frame_row["recording_name"]
    video_frame_idx = int(frame_row["video_frame_idx"])
    recording_id = NB04_TO_RECORDING_ID[recording_name]
    native_metadata = NATIVE_METADATA_BY_RECORDING_ID[recording_id]

    result_key = (recording_name, video_frame_idx)
    cached_baseline_waveform = BASELINE_FRAME_RESULTS.get(result_key)

    baseline_metric = derive_max_velocity_cm_s_from_waveform(
        waveform_result=cached_baseline_waveform,
        cm_s_per_px=float(frame_row["cm_s_per_px"]),
        roi_y_min=MANUAL_ROI["y_min"],
    )

    native_roi = native_roi_from_metadata(native_metadata)

    if result_key not in NATIVE_ROI_FRAME_RESULTS:
        NATIVE_ROI_FRAME_RESULTS[result_key] = extract_waveform_for_frame_row(frame_row, native_roi)

    native_roi_waveform = NATIVE_ROI_FRAME_RESULTS[result_key]

    candidate_metric = derive_max_velocity_cm_s_from_waveform(
        waveform_result=native_roi_waveform,
        cm_s_per_px=float(native_metadata["cm_s_per_px"]),
        roi_y_min=native_roi["y_min"],
    )

    delta_cm_s = candidate_metric["max_velocity_cm_s"] - baseline_metric["max_velocity_cm_s"]

    roi_plus_scale_rows.append(
        {
            "recording_name": recording_name,
            "recording_id": recording_id,
            "video_frame_idx": video_frame_idx,
            "native_class": PROJECT_NATIVE_QC_CLASS.get(recording_id, "unclassified"),
            "baseline_velocity_cm_s": baseline_metric["max_velocity_cm_s"],
            "native_roi_scale_velocity_cm_s": candidate_metric["max_velocity_cm_s"],
            "dvel_cm_s": delta_cm_s,
            "abs_dvel_cm_s": abs(delta_cm_s),
            "manual_cm_s_per_px": float(frame_row["cm_s_per_px"]),
            "native_cm_s_per_px": float(native_metadata["cm_s_per_px"]),
            "baseline_direction": cached_baseline_waveform.get("direction") if cached_baseline_waveform else None,
            "native_roi_direction": native_roi_waveform.get("direction"),
        }
    )

native_roi_plus_scale_delta = pd.DataFrame(roi_plus_scale_rows)

roi_plus_scale_csv_path = NB13_OUT_DIR / "nb13_delta_native_roi_plus_scale.csv"
native_roi_plus_scale_delta.to_csv(roi_plus_scale_csv_path, index=False)

roi_plus_scale_summary = (
    native_roi_plus_scale_delta
    .groupby(["recording_name", "recording_id"])
    .agg(
        n_frames=("dvel_cm_s", "size"),
        median_dvel_cm_s=("dvel_cm_s", "median"),
        median_abs_dvel_cm_s=("abs_dvel_cm_s", "median"),
        max_abs_dvel_cm_s=("abs_dvel_cm_s", "max"),
        frames_changed_gt_0p05=("abs_dvel_cm_s", lambda values: int((values > 0.05).sum())),
        direction_changes=("native_roi_direction", lambda values: int((values != native_roi_plus_scale_delta.loc[values.index, "baseline_direction"]).sum())),
    )
    .reset_index()
)

print("Native ROI plus scale calibration delta")
print("-" * 88)
print(f"mapped frames evaluated:           {len(native_roi_plus_scale_delta)}")
print(f"median dvel cm/s:                  {native_roi_plus_scale_delta['dvel_cm_s'].median():.6f}")
print(f"median absolute dvel cm/s:         {native_roi_plus_scale_delta['abs_dvel_cm_s'].median():.6f}")
print(f"max absolute dvel cm/s:            {native_roi_plus_scale_delta['abs_dvel_cm_s'].max():.6f}")
print(f"frames changed > 0.05 cm/s:        {int((native_roi_plus_scale_delta['abs_dvel_cm_s'] > 0.05).sum())}")
print(f"cached native ROI frame results:   {len(NATIVE_ROI_FRAME_RESULTS)}")
print(f"saved delta CSV:                   {roi_plus_scale_csv_path}")
print("-" * 88)

display(roi_plus_scale_summary)

if len(native_roi_plus_scale_delta) != 380:
    raise ValueError(f"Expected 380 mapped frames, found {len(native_roi_plus_scale_delta)}")

if native_roi_plus_scale_delta["dvel_cm_s"].isna().any():
    raise ValueError("At least one native ROI plus scale delta is missing.")

Native ROI plus scale calibration delta
----------------------------------------------------------------------------------------
mapped frames evaluated:           380
median dvel cm/s:                  0.166854
median absolute dvel cm/s:         0.167030
max absolute dvel cm/s:            3.788662
frames changed > 0.05 cm/s:        376
cached native ROI frame results:   380
saved delta CSV:                   D:\code\DopplerLab\feature_exports\nb13_validation\nb13_delta_native_roi_plus_scale.csv
----------------------------------------------------------------------------------------


,recording_name,recording_id,n_frames,median_dvel_cm_s,median_abs_dvel_cm_s,max_abs_dvel_cm_s,frames_changed_gt_0p05,direction_changes
0,candidate_test_01_brachial,202606130437480012SMP,34,0.116374,0.116374,0.278466,34,0
1,candidate_test_02_brachial,202606130411060002SMP,54,0.183206,0.183889,1.457027,54,0
2,candidate_test_02_neck,202606130422440006SMP,49,0.139829,0.146616,2.646691,49,2
3,candidate_test_03_brachial,202606130413540003SMP,43,0.167118,0.169364,0.183300,42,0
4,candidate_test_04_brachial_1,202606130434130010SMP,55,0.166898,0.166898,0.805232,53,0
5,candidate_test_04_brachial_2,202606130433280009SMP,54,0.165133,0.165281,3.788662,53,2
6,candidate_test_05_brachial,202606130426500007SMP,45,0.213253,0.213253,2.936571,45,0
7,candidate_test_08_brachial,202606130420260005SMP,46,0.182600,0.184661,1.608798,46,0


## Native ROI plus scale plus baseline calibration delta

### What this tests

This section tests the candidate effect of using the native `DcmRegionPara` ROI, native velocity scale, and native zero-velocity baseline override together.

### Why this matters

This is the full native metadata-derived calibration candidate. It combines uniform scale effects, ROI-edge effects, and direction-aware baseline re-reference effects.


In [26]:
roi_scale_baseline_rows = []

for _, frame_row in mapped_frames.iterrows():
    recording_name = frame_row["recording_name"]
    video_frame_idx = int(frame_row["video_frame_idx"])
    recording_id = NB04_TO_RECORDING_ID[recording_name]
    native_metadata = NATIVE_METADATA_BY_RECORDING_ID[recording_id]

    result_key = (recording_name, video_frame_idx)
    cached_baseline_waveform = BASELINE_FRAME_RESULTS.get(result_key)

    baseline_metric = derive_max_velocity_cm_s_from_waveform(
        waveform_result=cached_baseline_waveform,
        cm_s_per_px=float(frame_row["cm_s_per_px"]),
        roi_y_min=MANUAL_ROI["y_min"],
    )

    native_roi = native_roi_from_metadata(native_metadata)

    if result_key not in NATIVE_ROI_FRAME_RESULTS:
        NATIVE_ROI_FRAME_RESULTS[result_key] = extract_waveform_for_frame_row(frame_row, native_roi)

    native_roi_waveform = NATIVE_ROI_FRAME_RESULTS[result_key]

    candidate_metric = derive_max_velocity_cm_s_from_waveform(
        waveform_result=native_roi_waveform,
        cm_s_per_px=float(native_metadata["cm_s_per_px"]),
        roi_y_min=native_roi["y_min"],
        native_baseline_global=float(native_metadata["baseline_y_global"]),
    )

    delta_cm_s = candidate_metric["max_velocity_cm_s"] - baseline_metric["max_velocity_cm_s"]

    roi_scale_baseline_rows.append(
        {
            "recording_name": recording_name,
            "recording_id": recording_id,
            "video_frame_idx": video_frame_idx,
            "native_class": PROJECT_NATIVE_QC_CLASS.get(recording_id, "unclassified"),
            "baseline_velocity_cm_s": baseline_metric["max_velocity_cm_s"],
            "native_roi_scale_baseline_velocity_cm_s": candidate_metric["max_velocity_cm_s"],
            "dvel_cm_s": delta_cm_s,
            "abs_dvel_cm_s": abs(delta_cm_s),
            "manual_cm_s_per_px": float(frame_row["cm_s_per_px"]),
            "native_cm_s_per_px": float(native_metadata["cm_s_per_px"]),
            "baseline_direction": cached_baseline_waveform.get("direction") if cached_baseline_waveform else None,
            "native_roi_direction": native_roi_waveform.get("direction"),
            "native_roi_auto_baseline_roi_px": native_roi_waveform.get("baseline"),
            "native_baseline_roi_px": float(native_metadata["baseline_y_global"]) - float(native_roi["y_min"]),
            "baseline_shift_px_applied": candidate_metric["baseline_shift_px"],
        }
    )

native_roi_scale_baseline_delta = pd.DataFrame(roi_scale_baseline_rows)

roi_scale_baseline_csv_path = NB13_OUT_DIR / "nb13_delta_native_roi_scale_baseline.csv"
native_roi_scale_baseline_delta.to_csv(roi_scale_baseline_csv_path, index=False)

roi_scale_baseline_summary = (
    native_roi_scale_baseline_delta
    .groupby(["recording_name", "recording_id"])
    .agg(
        n_frames=("dvel_cm_s", "size"),
        median_dvel_cm_s=("dvel_cm_s", "median"),
        median_abs_dvel_cm_s=("abs_dvel_cm_s", "median"),
        max_abs_dvel_cm_s=("abs_dvel_cm_s", "max"),
        frames_changed_gt_0p05=("abs_dvel_cm_s", lambda values: int((values > 0.05).sum())),
        direction_changes=("native_roi_direction", lambda values: int((values != native_roi_scale_baseline_delta.loc[values.index, "baseline_direction"]).sum())),
        min_shift_px_applied=("baseline_shift_px_applied", "min"),
        max_shift_px_applied=("baseline_shift_px_applied", "max"),
    )
    .reset_index()
)

print("Native ROI plus scale plus baseline calibration delta")
print("-" * 88)
print(f"mapped frames evaluated:           {len(native_roi_scale_baseline_delta)}")
print(f"median dvel cm/s:                  {native_roi_scale_baseline_delta['dvel_cm_s'].median():.6f}")
print(f"median absolute dvel cm/s:         {native_roi_scale_baseline_delta['abs_dvel_cm_s'].median():.6f}")
print(f"max absolute dvel cm/s:            {native_roi_scale_baseline_delta['abs_dvel_cm_s'].max():.6f}")
print(f"frames changed > 0.05 cm/s:        {int((native_roi_scale_baseline_delta['abs_dvel_cm_s'] > 0.05).sum())}")
print(f"cached native ROI frame results:   {len(NATIVE_ROI_FRAME_RESULTS)}")
print(f"saved delta CSV:                   {roi_scale_baseline_csv_path}")
print("-" * 88)

display(roi_scale_baseline_summary)

if len(native_roi_scale_baseline_delta) != 380:
    raise ValueError(f"Expected 380 mapped frames, found {len(native_roi_scale_baseline_delta)}")

if native_roi_scale_baseline_delta["dvel_cm_s"].isna().any():
    raise ValueError("At least one native ROI plus scale plus baseline delta is missing.")

Native ROI plus scale plus baseline calibration delta
----------------------------------------------------------------------------------------
mapped frames evaluated:           380
median dvel cm/s:                  0.166583
median absolute dvel cm/s:         0.177652
max absolute dvel cm/s:            5.123931
frames changed > 0.05 cm/s:        377
cached native ROI frame results:   380
saved delta CSV:                   D:\code\DopplerLab\feature_exports\nb13_validation\nb13_delta_native_roi_scale_baseline.csv
----------------------------------------------------------------------------------------


,recording_name,recording_id,n_frames,median_dvel_cm_s,median_abs_dvel_cm_s,max_abs_dvel_cm_s,frames_changed_gt_0p05,direction_changes,min_shift_px_applied,max_shift_px_applied
0,candidate_test_01_brachial,202606130437480012SMP,34,0.116374,0.116374,0.278466,34,0,0.0,0.0
1,candidate_test_02_brachial,202606130411060002SMP,54,0.183206,0.183889,1.457027,54,0,0.0,0.0
2,candidate_test_02_neck,202606130422440006SMP,49,0.158186,1.352559,5.123931,49,2,0.0,15.0
3,candidate_test_03_brachial,202606130413540003SMP,43,0.167118,0.169364,0.183300,42,0,0.0,0.0
4,candidate_test_04_brachial_1,202606130434130010SMP,55,0.166898,0.166898,3.360227,53,0,-20.0,0.0
5,candidate_test_04_brachial_2,202606130433280009SMP,54,-0.277915,2.460507,3.788662,54,2,-22.0,0.0
6,candidate_test_05_brachial,202606130426500007SMP,45,0.213253,0.213253,2.936571,45,0,0.0,0.0
7,candidate_test_08_brachial,202606130420260005SMP,46,0.182600,0.184661,1.608798,46,0,0.0,0.0


## Baseline override diagnostics by recording and direction

### What this tests

This section tests where the auto-detected baseline differs from the native metadata baseline, grouped by recording and spectrum direction.

### Why this matters

Native baseline override is direction-aware. This diagnostic table explains which recordings produce nonzero baseline shifts and whether those shifts occur in spectra above or below the baseline.


In [27]:
baseline_override_diagnostics = (
    native_baseline_override_delta
    .groupby(["recording_name", "recording_id", "direction"])
    .agg(
        n_frames=("auto_minus_native_px", "size"),
        n_nonzero_auto_native_baseline_delta=(
            "auto_minus_native_px",
            lambda values: int((values.abs() > 0.5).sum()),
        ),
        min_auto_minus_native_px=("auto_minus_native_px", "min"),
        max_auto_minus_native_px=("auto_minus_native_px", "max"),
        min_shift_px_applied=("baseline_shift_px_applied", "min"),
        max_shift_px_applied=("baseline_shift_px_applied", "max"),
        max_abs_dvel_cm_s=("abs_dvel_cm_s", "max"),
    )
    .round(3)
    .reset_index()
)

diagnostics_csv_path = NB13_OUT_DIR / "nb13_baseline_override_diagnostics.csv"
baseline_override_diagnostics.to_csv(diagnostics_csv_path, index=False)

nonzero_diagnostics = baseline_override_diagnostics[
    baseline_override_diagnostics["n_nonzero_auto_native_baseline_delta"] > 0
].copy()

print("Baseline override diagnostics by recording and direction")
print("-" * 88)
print(f"diagnostic groups:                 {len(baseline_override_diagnostics)}")
print(f"groups with nonzero shift:         {len(nonzero_diagnostics)}")
print(f"frames with nonzero shift:         {int(baseline_override_diagnostics['n_nonzero_auto_native_baseline_delta'].sum())}")
print(f"saved diagnostics CSV:             {diagnostics_csv_path}")
print("-" * 88)

display(baseline_override_diagnostics)

print("Groups with nonzero auto/native baseline difference")
print("-" * 88)
display(nonzero_diagnostics)

if len(nonzero_diagnostics) == 0:
    raise ValueError("Expected at least one nonzero native baseline diagnostic group.")

Baseline override diagnostics by recording and direction
----------------------------------------------------------------------------------------
diagnostic groups:                 12
groups with nonzero shift:         4
frames with nonzero shift:         51
saved diagnostics CSV:             D:\code\DopplerLab\feature_exports\nb13_validation\nb13_baseline_override_diagnostics.csv
----------------------------------------------------------------------------------------


,recording_name,recording_id,direction,n_frames,n_nonzero_auto_native_baseline_delta,min_auto_minus_native_px,max_auto_minus_native_px,min_shift_px_applied,max_shift_px_applied,max_abs_dvel_cm_s
0,candidate_test_01_brachial,202606130437480012SMP,below,34,0,0.0,0.0,0.0,0.0,0.000
1,candidate_test_02_brachial,202606130411060002SMP,below,54,0,0.0,0.0,0.0,0.0,0.000
2,candidate_test_02_neck,202606130422440006SMP,above,47,20,-15.0,0.0,0.0,15.0,2.632
3,candidate_test_02_neck,202606130422440006SMP,below,2,2,-15.0,-14.0,-15.0,-14.0,2.632
4,candidate_test_03_brachial,202606130413540003SMP,above,43,0,0.0,0.0,0.0,0.0,0.000
5,candidate_test_04_brachial_1,202606130434130010SMP,above,53,0,0.0,0.0,0.0,0.0,0.000
6,candidate_test_04_brachial_1,202606130434130010SMP,below,2,2,-20.0,-20.0,-20.0,-20.0,3.509
7,candidate_test_04_brachial_2,202606130433280009SMP,above,22,0,0.0,0.0,0.0,0.0,0.000
8,candidate_test_04_brachial_2,202606130433280009SMP,below,32,27,-22.0,0.0,-22.0,0.0,3.860
9,candidate_test_05_brachial,202606130426500007SMP,above,30,0,0.0,0.0,0.0,0.0,0.000


Groups with nonzero auto/native baseline difference
----------------------------------------------------------------------------------------


,recording_name,recording_id,direction,n_frames,n_nonzero_auto_native_baseline_delta,min_auto_minus_native_px,max_auto_minus_native_px,min_shift_px_applied,max_shift_px_applied,max_abs_dvel_cm_s
2,candidate_test_02_neck,202606130422440006SMP,above,47,20,-15.0,0.0,0.0,15.0,2.632
3,candidate_test_02_neck,202606130422440006SMP,below,2,2,-15.0,-14.0,-15.0,-14.0,2.632
6,candidate_test_04_brachial_1,202606130434130010SMP,below,2,2,-20.0,-20.0,-20.0,-20.0,3.509
8,candidate_test_04_brachial_2,202606130433280009SMP,below,32,27,-22.0,0.0,-22.0,0.0,3.860


## Native calibration delta summary checkpoint

### What this tests

This section summarizes the split native calibration candidate experiments: scale-only, ROI-only, baseline-override-only, ROI plus scale, and ROI plus scale plus baseline.

### Why this matters

The split summary shows which metadata-derived component drives each velocity change. It keeps uniform scale effects separate from ROI-edge effects and baseline re-reference effects.



In [28]:
calibration_delta_tables = {
    "native_scale_only": native_scale_only_delta,
    "native_roi_only": native_roi_only_delta,
    "native_baseline_override_only": native_baseline_override_delta,
    "native_roi_plus_scale": native_roi_plus_scale_delta,
    "native_roi_scale_baseline": native_roi_scale_baseline_delta,
}

summary_rows = []

for experiment_name, delta_table in calibration_delta_tables.items():
    summary_rows.append(
        {
            "experiment": experiment_name,
            "n": len(delta_table),
            "median_dvel_cm_s": round(float(delta_table["dvel_cm_s"].median()), 3),
            "median_abs_dvel_cm_s": round(float(delta_table["abs_dvel_cm_s"].median()), 3),
            "max_abs_dvel_cm_s": round(float(delta_table["abs_dvel_cm_s"].max()), 3),
            "frames_changed_gt_0p05": int((delta_table["abs_dvel_cm_s"] > 0.05).sum()),
        }
    )

native_calibration_delta_summary = pd.DataFrame(summary_rows)

print("Native calibration delta summary checkpoint")
print("-" * 88)
display(native_calibration_delta_summary)

expected_n = {
    "native_scale_only": 380,
    "native_roi_only": 380,
    "native_baseline_override_only": 380,
    "native_roi_plus_scale": 380,
    "native_roi_scale_baseline": 380,
}

for experiment_name, expected_rows in expected_n.items():
    observed_rows = int(
        native_calibration_delta_summary.loc[
            native_calibration_delta_summary["experiment"] == experiment_name,
            "n",
        ].iloc[0]
    )

    if observed_rows != expected_rows:
        raise ValueError(
            f"{experiment_name} expected {expected_rows} rows, found {observed_rows}"
        )

Native calibration delta summary checkpoint
----------------------------------------------------------------------------------------


,experiment,n,median_dvel_cm_s,median_abs_dvel_cm_s,max_abs_dvel_cm_s,frames_changed_gt_0p05
0,native_scale_only,380,0.165,0.165,0.219,380
1,native_roi_only,380,0.000,0.000,3.645,35
2,native_baseline_override_only,380,0.000,0.000,3.860,51
3,native_roi_plus_scale,380,0.167,0.167,3.789,376
4,native_roi_scale_baseline,380,0.167,0.178,5.124,377


### Interpretation - native calibration delta summary

**Result classification:** split native calibration checkpoint passed.

The split calibration experiments separate the native metadata effects. Native scale-only produces a small uniform positive velocity shift. Native ROI-only has no median effect but produces localized frame-level edge effects. Native baseline-override-only has no median effect but produces localized direction-aware re-reference effects. The combined candidates compound these behaviors.

The result supports treating native metadata as a reproducible candidate calibration source.

The result does not support changing the baseline pipeline default. Native ROI and baseline override effects require visual review before trust.

The next working hypothesis is that robustness checks can identify candidate frame-level failure modes without changing the default image pipeline.

## Sweep-marker mask candidate delta

### What this tests

This section tests whether trimming a small lower band from the manual ROI changes extracted frame-level max velocity relative to the validated baseline.

### Why this matters

Pixels near the lower ROI edge can include sweep-marker or display artifacts. A simple trim may reduce some spurious high envelope detections, but it may also remove real signal near the baseline.


In [29]:
sweep_marker_rows = []

for _, frame_row in mapped_frames.iterrows():
    recording_name = frame_row["recording_name"]
    video_frame_idx = int(frame_row["video_frame_idx"])
    recording_id = NB04_TO_RECORDING_ID[recording_name]
    native_metadata = NATIVE_METADATA_BY_RECORDING_ID[recording_id]

    result_key = (recording_name, video_frame_idx)
    cached_baseline_waveform = BASELINE_FRAME_RESULTS.get(result_key)

    baseline_metric = derive_max_velocity_cm_s_from_waveform(
        waveform_result=cached_baseline_waveform,
        cm_s_per_px=float(frame_row["cm_s_per_px"]),
        roi_y_min=MANUAL_ROI["y_min"],
    )

    sweep_mask_roi = dict(MANUAL_ROI)
    sweep_mask_roi["y_max"] = min(
        int(sweep_mask_roi["y_max"]),
        int(native_metadata["y1"]) - 3,
    )

    sweep_mask_waveform = extract_waveform_for_frame_row(frame_row, sweep_mask_roi)

    candidate_metric = derive_max_velocity_cm_s_from_waveform(
        waveform_result=sweep_mask_waveform,
        cm_s_per_px=float(frame_row["cm_s_per_px"]),
        roi_y_min=sweep_mask_roi["y_min"],
    )

    delta_cm_s = candidate_metric["max_velocity_cm_s"] - baseline_metric["max_velocity_cm_s"]

    sweep_marker_rows.append(
        {
            "recording_name": recording_name,
            "recording_id": recording_id,
            "video_frame_idx": video_frame_idx,
            "native_class": PROJECT_NATIVE_QC_CLASS.get(recording_id, "unclassified"),
            "baseline_velocity_cm_s": baseline_metric["max_velocity_cm_s"],
            "sweep_mask_velocity_cm_s": candidate_metric["max_velocity_cm_s"],
            "dvel_cm_s": delta_cm_s,
            "abs_dvel_cm_s": abs(delta_cm_s),
            "manual_y_max": MANUAL_ROI["y_max"],
            "sweep_mask_y_max": sweep_mask_roi["y_max"],
            "baseline_direction": cached_baseline_waveform.get("direction") if cached_baseline_waveform else None,
            "sweep_mask_direction": sweep_mask_waveform.get("direction"),
            "baseline_auto_baseline_roi_px": cached_baseline_waveform.get("baseline") if cached_baseline_waveform else np.nan,
            "sweep_mask_auto_baseline_roi_px": sweep_mask_waveform.get("baseline"),
        }
    )

sweep_marker_delta = pd.DataFrame(sweep_marker_rows)

sweep_marker_csv_path = NB13_OUT_DIR / "nb13_delta_sweep_marker_mask.csv"
sweep_marker_delta.to_csv(sweep_marker_csv_path, index=False)

sweep_marker_summary = (
    sweep_marker_delta
    .groupby(["recording_name", "recording_id"])
    .agg(
        n_frames=("dvel_cm_s", "size"),
        median_dvel_cm_s=("dvel_cm_s", "median"),
        median_abs_dvel_cm_s=("abs_dvel_cm_s", "median"),
        max_abs_dvel_cm_s=("abs_dvel_cm_s", "max"),
        frames_changed_gt_0p05=("abs_dvel_cm_s", lambda values: int((values > 0.05).sum())),
        direction_changes=("sweep_mask_direction", lambda values: int((values != sweep_marker_delta.loc[values.index, "baseline_direction"]).sum())),
    )
    .reset_index()
)

print("Sweep-marker mask candidate delta")
print("-" * 88)
print(f"mapped frames evaluated:           {len(sweep_marker_delta)}")
print(f"median dvel cm/s:                  {sweep_marker_delta['dvel_cm_s'].median():.6f}")
print(f"median absolute dvel cm/s:         {sweep_marker_delta['abs_dvel_cm_s'].median():.6f}")
print(f"max absolute dvel cm/s:            {sweep_marker_delta['abs_dvel_cm_s'].max():.6f}")
print(f"frames changed > 0.05 cm/s:        {int((sweep_marker_delta['abs_dvel_cm_s'] > 0.05).sum())}")
print(f"saved delta CSV:                   {sweep_marker_csv_path}")
print("-" * 88)

display(sweep_marker_summary)

if len(sweep_marker_delta) != 380:
    raise ValueError(f"Expected 380 mapped frames, found {len(sweep_marker_delta)}")

if sweep_marker_delta["dvel_cm_s"].isna().any():
    raise ValueError("At least one sweep-marker mask delta is missing.")

Sweep-marker mask candidate delta
----------------------------------------------------------------------------------------
mapped frames evaluated:           380
median dvel cm/s:                  0.000000
median absolute dvel cm/s:         0.000000
max absolute dvel cm/s:            1.675427
frames changed > 0.05 cm/s:        58
saved delta CSV:                   D:\code\DopplerLab\feature_exports\nb13_validation\nb13_delta_sweep_marker_mask.csv
----------------------------------------------------------------------------------------


,recording_name,recording_id,n_frames,median_dvel_cm_s,median_abs_dvel_cm_s,max_abs_dvel_cm_s,frames_changed_gt_0p05,direction_changes
0,candidate_test_01_brachial,202606130437480012SMP,34,0.0,0.0,1.182746,2,0
1,candidate_test_02_brachial,202606130411060002SMP,54,0.0,0.0,0.599481,20,0
2,candidate_test_02_neck,202606130422440006SMP,49,0.0,0.0,0.000000,0,0
3,candidate_test_03_brachial,202606130413540003SMP,43,0.0,0.0,0.000000,0,0
4,candidate_test_04_brachial_1,202606130434130010SMP,55,0.0,0.0,0.000000,0,0
5,candidate_test_04_brachial_2,202606130433280009SMP,54,0.0,0.0,1.675427,5,0
6,candidate_test_05_brachial,202606130426500007SMP,45,0.0,0.0,0.904412,15,0
7,candidate_test_08_brachial,202606130420260005SMP,46,0.0,0.0,1.010601,16,0


## Autocorrelation-guided beat detector candidate

### What this tests

This section tests whether an autocorrelation-guided candidate beat detector changes peak counts and recomputed complete-beat counts relative to the current detector.

### Why this matters

A candidate detector can reduce or increase peak counts, but the meaningful image-path effect is whether complete beat morphology changes after peaks are replaced and morphology is recomputed.


In [30]:
#| export waveform
from scipy.signal import find_peaks


def autocorr_guided_peaks(velocity_smooth, min_cycle_length_px=60, max_cycle_length_px=400):
    """
    This function proposes candidate peaks using an autocorrelation-derived
    cycle-length prior.

    It is an experimental beat detector candidate, not the default detector.
    """
    velocity_smooth = np.asarray(velocity_smooth, dtype=float)

    if velocity_smooth.ndim != 1:
        raise ValueError("velocity_smooth must be a 1D array.")

    centered = velocity_smooth - np.mean(velocity_smooth)

    if centered.std() == 0:
        return np.array([], dtype=int)

    n_samples = len(centered)

    if n_samples <= min_cycle_length_px + 5:
        fallback_peaks, _ = find_peaks(
            velocity_smooth,
            height=80,
            distance=100,
        )
        return fallback_peaks

    fft_values = np.fft.rfft(centered, n=2 * n_samples)
    autocorr = np.fft.irfft(fft_values * np.conj(fft_values))[:n_samples]
    autocorr = autocorr / autocorr[0]

    search_stop = min(n_samples - 1, max_cycle_length_px)

    if search_stop <= min_cycle_length_px:
        fallback_peaks, _ = find_peaks(
            velocity_smooth,
            height=80,
            distance=100,
        )
        return fallback_peaks

    cycle_length_px = min_cycle_length_px + int(
        np.argmax(autocorr[min_cycle_length_px:search_stop])
    )

    candidate_distance = max(1, int(0.7 * cycle_length_px))

    candidate_peaks, _ = find_peaks(
        velocity_smooth,
        height=80,
        distance=candidate_distance,
    )

    return candidate_peaks


def analyze_waveform_with_replaced_peaks(waveform_result, peaks_local):
    """
    This function recomputes beat morphology after replacing the detected peaks.
    """
    replaced = dict(waveform_result)
    replaced["peaks_local"] = np.asarray(peaks_local, dtype=int)
    replaced["peaks_global"] = np.asarray(peaks_local, dtype=int) + int(waveform_result["x_offset"])
    replaced["peaks"] = np.asarray(peaks_local, dtype=int)

    if len(replaced["peaks_local"]) < 2:
        return pd.DataFrame()

    return analyze_beats(replaced)

In [31]:
beat_detector_rows = []

for _, frame_row in frames_ok.iterrows():
    recording_name = frame_row["recording_name"]
    video_frame_idx = int(frame_row["video_frame_idx"])
    result_key = (recording_name, video_frame_idx)

    cached_waveform = BASELINE_FRAME_RESULTS.get(result_key)

    if cached_waveform is None:
        continue

    current_peaks = np.asarray(cached_waveform["peaks_local"], dtype=int)
    candidate_peaks = autocorr_guided_peaks(cached_waveform["velocity_smooth"])

    current_beat_table = analyze_waveform_with_replaced_peaks(
        waveform_result=cached_waveform,
        peaks_local=current_peaks,
    )

    candidate_beat_table = analyze_waveform_with_replaced_peaks(
        waveform_result=cached_waveform,
        peaks_local=candidate_peaks,
    )

    current_complete = (
        int(current_beat_table["is_complete_beat"].sum())
        if len(current_beat_table) > 0 and "is_complete_beat" in current_beat_table
        else 0
    )

    candidate_complete = (
        int(candidate_beat_table["is_complete_beat"].sum())
        if len(candidate_beat_table) > 0 and "is_complete_beat" in candidate_beat_table
        else 0
    )

    beat_detector_rows.append(
        {
            "recording_name": recording_name,
            "video_frame_idx": video_frame_idx,
            "current_peaks": int(len(current_peaks)),
            "candidate_peaks": int(len(candidate_peaks)),
            "current_total_beats": int(len(current_beat_table)),
            "candidate_total_beats": int(len(candidate_beat_table)),
            "current_complete_beats": current_complete,
            "candidate_complete_beats": candidate_complete,
            "dpeaks": int(len(candidate_peaks) - len(current_peaks)),
            "d_total_beats": int(len(candidate_beat_table) - len(current_beat_table)),
            "d_complete_beats": int(candidate_complete - current_complete),
        }
    )

autocorr_beat_detector_delta = pd.DataFrame(beat_detector_rows)

beat_detector_csv_path = NB13_OUT_DIR / "nb13_delta_autocorr_beat_detector.csv"
autocorr_beat_detector_delta.to_csv(beat_detector_csv_path, index=False)

beat_detector_summary = (
    autocorr_beat_detector_delta
    .groupby("recording_name")
    .agg(
        n_frames=("dpeaks", "size"),
        total_dpeaks=("dpeaks", "sum"),
        total_d_complete_beats=("d_complete_beats", "sum"),
        frames_with_peak_count_change=("dpeaks", lambda values: int((values != 0).sum())),
        frames_with_complete_beat_change=("d_complete_beats", lambda values: int((values != 0).sum())),
    )
    .reset_index()
)

print("Autocorrelation-guided beat detector candidate")
print("-" * 88)
print(f"frames evaluated:                  {len(autocorr_beat_detector_delta)}")
print(f"total dpeaks:                      {int(autocorr_beat_detector_delta['dpeaks'].sum())}")
print(f"total d_complete_beats:            {int(autocorr_beat_detector_delta['d_complete_beats'].sum())}")
print(f"frames with peak count change:     {int((autocorr_beat_detector_delta['dpeaks'] != 0).sum())}")
print(f"frames with complete-beat change:  {int((autocorr_beat_detector_delta['d_complete_beats'] != 0).sum())}")
print(f"saved detector CSV:                {beat_detector_csv_path}")
print("-" * 88)

display(beat_detector_summary)

if len(autocorr_beat_detector_delta) != 406:
    raise ValueError(f"Expected 406 beat-detector rows, found {len(autocorr_beat_detector_delta)}")

Autocorrelation-guided beat detector candidate
----------------------------------------------------------------------------------------
frames evaluated:                  406
total dpeaks:                      185
total d_complete_beats:            125
frames with peak count change:     130
frames with complete-beat change:  103
saved detector CSV:                D:\code\DopplerLab\feature_exports\nb13_validation\nb13_delta_autocorr_beat_detector.csv
----------------------------------------------------------------------------------------


,recording_name,n_frames,total_dpeaks,total_d_complete_beats,frames_with_peak_count_change,frames_with_complete_beat_change
0,candidate_test_01_brachial,34,0,0,0,0
1,candidate_test_02_brachial,54,0,0,0,0
2,candidate_test_02_neck,49,0,0,0,0
3,candidate_test_03_brachial,43,0,0,0,0
4,candidate_test_04_brachial_1,55,101,66,50,41
5,candidate_test_04_brachial_2,54,100,71,47,34
6,candidate_test_04_neck,26,0,0,0,0
7,candidate_test_05_brachial,45,-16,-12,33,28
8,candidate_test_08_brachial,46,0,0,0,0


### Interpretation - autocorrelation-guided beat detector candidate

**Result classification:** candidate beat-detector behavior measured; not morphology-validated.

The autocorrelation-guided detector changes peak counts and recomputed complete-beat counts in selected recordings. The effect is concentrated in `candidate_test_04_brachial_1`, `candidate_test_04_brachial_2`, and `candidate_test_05_brachial`.

The result supports treating the detector as a candidate morphology experiment with measurable downstream effects.

The result does not support replacing the current detector. Peak-count and complete-beat changes require frame-level visual review before trust.

The next working hypothesis is that simple audio clipping metrics can reproduce the existing registry clipping flag as an additive QC signal.

## Audio clipping candidate QC

### What this tests

This section tests whether a simple audio clipping metric reproduces the registry `audio_possible_clipping` flag.

### Why this matters

Clipped audio may make audio-derived HR less reliable. A cheap additive QC flag can help interpret audio/native HR agreement later.


In [ ]:
#| export audio_qc
from pathlib import Path
import numpy as np
from scipy.io import wavfile



def audio_clipping_metrics(wav_path, near_peak_threshold=0.98, clip_fraction_threshold=0.001):
    """
    This function computes a candidate audio clipping metric from a WAV file.

    The output is an additive QC flag only. It does not change Doppler waveform
    extraction or beat logic.
    """
    wav_path = Path(wav_path)

    if not wav_path.exists():
        return {
            "audio_clip_fraction": np.nan,
            "audio_peak": np.nan,
            "audio_possible_clipping_candidate": False,
        }

    sample_rate_hz, audio = wavfile.read(wav_path)

    audio = audio.astype(float)

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    if len(audio) == 0:
        return {
            "audio_clip_fraction": np.nan,
            "audio_peak": np.nan,
            "audio_possible_clipping_candidate": False,
        }

    peak = float(np.max(np.abs(audio))) + 1e-9
    clip_fraction = float((np.abs(audio) >= near_peak_threshold * peak).mean())
    touches_int16_rail = peak >= 32767

    candidate_clip = bool(
        clip_fraction > clip_fraction_threshold
        and touches_int16_rail
    )

    return {
        "audio_clip_fraction": round(clip_fraction, 5),
        "audio_peak": int(round(peak)),
        "audio_possible_clipping_candidate": candidate_clip,
    }

In [33]:
import subprocess
import tempfile

registry_csv_path = (
    paths["feature_exports_dir"]
    / "nb04_v2_batch_2026_06_13"
    / "nb04_v2_accepted_recording_registry.csv"
)

if not registry_csv_path.exists():
    raise FileNotFoundError(f"Accepted recording registry not found: {registry_csv_path}")

recording_registry = pd.read_csv(registry_csv_path)

required_registry_columns = ["recording_name", "audio_possible_clipping", "mean_hr_bpm"]
missing_registry_columns = [
    column for column in required_registry_columns
    if column not in recording_registry.columns
]

if missing_registry_columns:
    raise ValueError(f"Registry is missing required columns: {missing_registry_columns}")

audio_clip_rows = []

for _, registry_row in recording_registry[required_registry_columns].iterrows():
    recording_name = registry_row["recording_name"]
    avi_path = paths["avi_batch_dir"] / f"{recording_name}.avi"

    if not avi_path.exists():
        continue

    wav_path = Path(tempfile.gettempdir()) / f"nb13_audio_clip_{recording_name}.wav"

    ffmpeg_result = subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-i",
            str(avi_path),
            "-ac",
            "1",
            str(wav_path),
        ],
        capture_output=True,
        text=True,
    )

    if ffmpeg_result.returncode == 0 and wav_path.exists():
        clipping = audio_clipping_metrics(wav_path)
        extraction_status = "audio_extracted"
    else:
        clipping = {
            "audio_clip_fraction": np.nan,
            "audio_peak": np.nan,
            "audio_possible_clipping_candidate": False,
        }
        extraction_status = "audio_extract_failed"

    audio_clip_rows.append(
        {
            "recording_name": recording_name,
            "registry_audio_possible_clipping": bool(registry_row["audio_possible_clipping"]),
            "candidate_audio_possible_clipping": clipping["audio_possible_clipping_candidate"],
            "audio_clip_fraction": clipping["audio_clip_fraction"],
            "audio_peak": clipping["audio_peak"],
            "mean_hr_bpm": registry_row["mean_hr_bpm"],
            "extraction_status": extraction_status,
        }
    )

audio_clipping_candidate = pd.DataFrame(audio_clip_rows)
audio_clipping_candidate["agrees_with_registry"] = (
    audio_clipping_candidate["registry_audio_possible_clipping"]
    == audio_clipping_candidate["candidate_audio_possible_clipping"]
)

audio_clipping_csv_path = NB13_OUT_DIR / "nb13_audio_clipping_candidate.csv"
audio_clipping_candidate.to_csv(audio_clipping_csv_path, index=False)

print("Audio clipping candidate QC")
print("-" * 88)
print(f"recordings evaluated:              {len(audio_clipping_candidate)}")
print(f"audio extraction failures:         {int((audio_clipping_candidate['extraction_status'] != 'audio_extracted').sum())}")
print(f"candidate agrees with registry:    {int(audio_clipping_candidate['agrees_with_registry'].sum())}/{len(audio_clipping_candidate)}")
print(f"candidate clipping positives:      {int(audio_clipping_candidate['candidate_audio_possible_clipping'].sum())}")
print(f"registry clipping positives:       {int(audio_clipping_candidate['registry_audio_possible_clipping'].sum())}")
print(f"saved audio clipping CSV:          {audio_clipping_csv_path}")
print("-" * 88)

display(audio_clipping_candidate)

if (audio_clipping_candidate["extraction_status"] != "audio_extracted").any():
    raise RuntimeError("At least one AVI audio extraction failed. Check ffmpeg availability and the extraction_status column.")

if not audio_clipping_candidate["agrees_with_registry"].all():
    raise ValueError("Candidate audio clipping flag did not reproduce the registry flag for every evaluated recording.")

Audio clipping candidate QC
----------------------------------------------------------------------------------------
recordings evaluated:              9
audio extraction failures:         0
candidate agrees with registry:    9/9
candidate clipping positives:      2
registry clipping positives:       2
saved audio clipping CSV:          D:\code\DopplerLab\feature_exports\nb13_validation\nb13_audio_clipping_candidate.csv
----------------------------------------------------------------------------------------


,recording_name,registry_audio_possible_clipping,candidate_audio_possible_clipping,audio_clip_fraction,audio_peak,mean_hr_bpm,extraction_status,agrees_with_registry
0,candidate_test_01_brachial,False,False,0.00006,30449,72.192513,audio_extracted,True
1,candidate_test_02_brachial,False,False,0.00000,29534,63.147668,audio_extracted,True
2,candidate_test_03_brachial,False,False,0.00000,7369,54.424779,audio_extracted,True
3,candidate_test_04_brachial_1,False,False,0.00002,18654,99.594246,audio_extracted,True
4,candidate_test_04_brachial_2,False,False,0.00002,21583,104.130283,audio_extracted,True
5,candidate_test_05_brachial,True,True,0.00154,32768,83.976834,audio_extracted,True
6,candidate_test_08_brachial,True,True,0.00351,32768,63.013443,audio_extracted,True
7,candidate_test_02_neck,False,False,0.00001,13161,58.489729,audio_extracted,True
8,candidate_test_04_neck,False,False,0.00011,18565,71.761531,audio_extracted,True


### Interpretation - audio clipping candidate QC

**Result classification:** audio clipping candidate QC reproduced registry flag.

The simple near-peak and int16-rail clipping metric reproduced the registry `audio_possible_clipping` flag for all evaluated recordings. The candidate positives are limited to recordings whose extracted audio reaches the int16 rail.

The result supports treating audio clipping as a cheap additive QC signal.

The result does not support changing image-path beat detection, native timing/QC extraction, or any clinical interpretation.

The next working hypothesis is that audio HR and native `hp_activity` HR can be compared as independent timing estimates, with missing audio handled as `not_assessed_missing_audio` rather than disagreement.

## Audio and native HR agreement

### What this tests

This section tests whether audio-derived HR and native `hp_activity` HR agree within 5 bpm for mapped native recordings.

### Why this matters

Audio HR and native `hp_activity` HR are independent timing estimates. Agreement can support recording-level QC, while disagreement or missing audio can flag recordings for review.


In [34]:
def classify_hr_agreement(abs_diff_bpm):
    """
    This function classifies agreement between audio HR and native HR.

    Missing audio is treated as not assessed, not as disagreement.
    """
    if pd.isna(abs_diff_bpm):
        return "not_assessed_missing_audio"

    if abs_diff_bpm <= 5.0:
        return "agree_5bpm"

    return "disagree_gt_5bpm"

In [35]:
registry_hr_by_recording = (
    recording_registry
    .set_index("recording_name")["mean_hr_bpm"]
    .to_dict()
)

native_hr_by_recording_id = (
    native_hp_sidecar
    .set_index("recording_id")["native_hr_bpm"]
    .to_dict()
)

hr_agreement_rows = []

for recording_id, nb04_recording_name in RECORDING_ID_TO_NB04.items():
    audio_hr_bpm = registry_hr_by_recording.get(nb04_recording_name, np.nan)
    native_hr_bpm = native_hr_by_recording_id.get(recording_id, np.nan)

    if pd.notna(audio_hr_bpm) and pd.notna(native_hr_bpm):
        abs_diff_bpm = abs(float(audio_hr_bpm) - float(native_hr_bpm))
    else:
        abs_diff_bpm = np.nan

    hr_agreement_rows.append(
        {
            "recording_id": recording_id,
            "nb04_recording_name": nb04_recording_name,
            "project_native_qc_class": PROJECT_NATIVE_QC_CLASS.get(recording_id, "unclassified"),
            "audio_hr_bpm": round(float(audio_hr_bpm), 1) if pd.notna(audio_hr_bpm) else np.nan,
            "native_hr_bpm": round(float(native_hr_bpm), 1) if pd.notna(native_hr_bpm) else np.nan,
            "abs_diff_bpm": round(float(abs_diff_bpm), 1) if pd.notna(abs_diff_bpm) else np.nan,
            "hr_agreement_status": classify_hr_agreement(abs_diff_bpm),
        }
    )

cross_modal_hr_agreement = pd.DataFrame(hr_agreement_rows).sort_values(
    ["project_native_qc_class", "recording_id"]
).reset_index(drop=True)

hr_agreement_csv_path = NB13_OUT_DIR / "nb13_cross_modal_hr_agreement.csv"
cross_modal_hr_agreement.to_csv(hr_agreement_csv_path, index=False)

status_counts = cross_modal_hr_agreement["hr_agreement_status"].value_counts().to_dict()
native_specific_rows = cross_modal_hr_agreement[
    cross_modal_hr_agreement["project_native_qc_class"] == "native_specific"
]

print("Audio and native HR agreement")
print("-" * 88)
print(f"mapped recordings evaluated:        {len(cross_modal_hr_agreement)}")
print(f"agreement status counts:            {status_counts}")
print(f"native_specific agree_5bpm:         {int((native_specific_rows['hr_agreement_status'] == 'agree_5bpm').sum())}")
print(f"native_specific assessed:           {int((native_specific_rows['hr_agreement_status'] != 'not_assessed_missing_audio').sum())}")
print(f"native_specific not assessed:       {int((native_specific_rows['hr_agreement_status'] == 'not_assessed_missing_audio').sum())}")
print(f"saved HR agreement CSV:             {hr_agreement_csv_path}")
print("-" * 88)

display(cross_modal_hr_agreement)

expected_status_counts = {
    "agree_5bpm": 5,
    "disagree_gt_5bpm": 3,
    "not_assessed_missing_audio": 2,
}

for status, expected_count in expected_status_counts.items():
    observed_count = int((cross_modal_hr_agreement["hr_agreement_status"] == status).sum())

    if observed_count != expected_count:
        raise ValueError(
            f"Expected {expected_count} rows with status {status}, found {observed_count}"
        )

Audio and native HR agreement
----------------------------------------------------------------------------------------
mapped recordings evaluated:        10
agreement status counts:            {'agree_5bpm': 5, 'disagree_gt_5bpm': 3, 'not_assessed_missing_audio': 2}
native_specific agree_5bpm:         4
native_specific assessed:           5
native_specific not assessed:       1
saved HR agreement CSV:             D:\code\DopplerLab\feature_exports\nb13_validation\nb13_cross_modal_hr_agreement.csv
----------------------------------------------------------------------------------------


,recording_id,nb04_recording_name,project_native_qc_class,audio_hr_bpm,native_hr_bpm,abs_diff_bpm,hr_agreement_status
0,202606130433280009SMP,candidate_test_04_brachial_2,audio_only,104.1,109.1,5.0,agree_5bpm
1,202606130434130010SMP,candidate_test_04_brachial_1,audio_only,99.6,107.1,7.5,disagree_gt_5bpm
2,202606130411060002SMP,candidate_test_02_brachial,native_specific,63.1,62.5,0.6,agree_5bpm
3,202606130413540003SMP,candidate_test_03_brachial,native_specific,54.4,54.1,0.3,agree_5bpm
4,202606130417060004SMP,candidate_test_06_brachial,native_specific,NaN,113.2,NaN,not_assessed_missing_audio
5,202606130420260005SMP,candidate_test_08_brachial,native_specific,63.0,64.5,1.5,agree_5bpm
6,202606130422440006SMP,candidate_test_02_neck,native_specific,58.5,58.3,0.2,agree_5bpm
7,202606130437480012SMP,candidate_test_01_brachial,native_specific,72.2,65.2,7.0,disagree_gt_5bpm
8,202606130426500007SMP,candidate_test_05_brachial,reject,84.0,64.5,19.5,disagree_gt_5bpm
9,202606130430380008SMP,candidate_test_07_brachial,reject,NaN,200.0,NaN,not_assessed_missing_audio


### Interpretation - audio and native HR agreement

**Result classification:** cross-modal HR QC checkpoint passed.

Audio-derived HR and native `hp_activity` HR agree within 5 bpm for five mapped recordings. Three assessed recordings disagree by more than 5 bpm, and two recordings are classified as `not_assessed_missing_audio`.

The result supports using audio/native HR agreement as an additive recording-level QC signal.

The result does not support clinical HR validation, native velocity-envelope recovery, or any diagnostic interpretation.

The next working hypothesis is that NB13 can close with an evidence-stage checkpoint that separates validated helpers, candidate improvements, review-needed items, and notebook-only experimental logic.

## Final evidence-stage checkpoint

### What this tests

This section summarizes the evidence stage for each NB13 candidate improvement and records whether the item is suitable for later nbdev modularization.

### Why this matters

NB13 is a validation bridge before package modularization. The final checkpoint separates stable helpers from candidate behavior, review-needed behavior, and notebook-only experimental logic.


In [36]:
native_calibration_delta_summary.to_csv(
    NB13_OUT_DIR / "nb13_flag_experiment_summary.csv",
    index=False,
)

nbdev_export_candidates = pd.DataFrame(
    [
        {
            "helper_or_group": "find_repo_root / dopplerlab_project_paths",
            "planned_module": "paths",
            "current_evidence": "path checks passed",
            "export_stage": "stable helper candidate",
        },
        {
            "helper_or_group": "read_video_frame_by_index_v2",
            "planned_module": "video",
            "current_evidence": "single-frame and full-frame baseline reproduction passed",
            "export_stage": "stable helper candidate",
        },
        {
            "helper_or_group": "extract_envelope_directional / extract_doppler_waveform / beat helpers",
            "planned_module": "waveform",
            "current_evidence": "full-frame baseline reproduction passed",
            "export_stage": "stable helper candidate",
        },
        {
            "helper_or_group": "fit_velocity_calibration_from_points / derive_max_velocity_cm_s_from_waveform",
            "planned_module": "calibration",
            "current_evidence": "baseline and split calibration deltas reproduced",
            "export_stage": "stable helper candidate",
        },
        {
            "helper_or_group": "build_native_to_avi_mapping",
            "planned_module": "native_mapping",
            "current_evidence": "10/10 native recordings matched with no ambiguity",
            "export_stage": "stable helper candidate",
        },
        {
            "helper_or_group": "parse_dcm_region_pw",
            "planned_module": "native_metadata",
            "current_evidence": "10/10 native metadata files parsed",
            "export_stage": "stable helper candidate",
        },
        {
            "helper_or_group": "native page/counter and hp_activity sidecar helpers",
            "planned_module": "native_qc",
            "current_evidence": "raw-byte sidecar reproduced and class agreement was 10/10",
            "export_stage": "experimental additive helper candidate",
        },
        {
            "helper_or_group": "audio_clipping_metrics",
            "planned_module": "audio_qc",
            "current_evidence": "registry clipping flag reproduced 9/9",
            "export_stage": "experimental additive helper candidate",
        },
        {
            "helper_or_group": "autocorr_guided_peaks",
            "planned_module": "waveform",
            "current_evidence": "candidate behavior measured, not visually validated",
            "export_stage": "keep candidate-labeled or notebook-only",
        },
    ]
)

nbdev_candidates_csv_path = NB13_OUT_DIR / "nb13_nbdev_export_candidates.csv"
nbdev_export_candidates.to_csv(nbdev_candidates_csv_path, index=False)

final_checkpoint = pd.DataFrame(
    [
        {
            "candidate_improvement": "Baseline harness equals NB04 V2 frame output",
            "evidence_reviewed": "nb13_baseline_reproduction_full.csv",
            "output_impact": "reference only",
            "risk": "none for validation anchor",
            "current_classification": "validated in full-frame harness",
            "later_nbdev_boundary": "export stable video/waveform/calibration helpers",
        },
        {
            "candidate_improvement": "Native-to-AVI mapping",
            "evidence_reviewed": "file-size mapping from native linked AVI files",
            "output_impact": "recording namespace bridge",
            "risk": "duplicate-size ambiguity if future data collide",
            "current_classification": "validated for current batch",
            "later_nbdev_boundary": "export stable native_mapping helper",
        },
        {
            "candidate_improvement": "Native DcmRegionPara metadata parsing",
            "evidence_reviewed": "nb13_native_metadata_calibration.csv",
            "output_impact": "metadata-derived ROI, baseline, scale, and time scale",
            "risk": "metadata is not proof of better calibration",
            "current_classification": "metadata-derived candidate input",
            "later_nbdev_boundary": "export parser, not defaults",
        },
        {
            "candidate_improvement": "Native scale-only calibration",
            "evidence_reviewed": "nb13_delta_native_scale_only.csv",
            "output_impact": "small uniform velocity rescale",
            "risk": "changes reported velocities",
            "current_classification": "candidate for later modularization",
            "later_nbdev_boundary": "export calculation support, keep default unchanged",
        },
        {
            "candidate_improvement": "Native ROI-only calibration",
            "evidence_reviewed": "nb13_delta_native_roi_only.csv",
            "output_impact": "localized ROI-edge velocity changes",
            "risk": "may add or remove signal/artifact pixels",
            "current_classification": "needs visual review",
            "later_nbdev_boundary": "do not make default",
        },
        {
            "candidate_improvement": "Native baseline override",
            "evidence_reviewed": "nb13_delta_native_baseline_override_only.csv and diagnostics",
            "output_impact": "localized direction-aware re-reference",
            "risk": "baseline shift changes all affected velocities",
            "current_classification": "needs visual review",
            "later_nbdev_boundary": "export helper logic only if kept clearly candidate-labeled",
        },
        {
            "candidate_improvement": "Native ROI plus scale plus baseline",
            "evidence_reviewed": "nb13_delta_native_roi_scale_baseline.csv",
            "output_impact": "combined metadata calibration delta",
            "risk": "compounds ROI, scale, and baseline effects",
            "current_classification": "candidate, not default",
            "later_nbdev_boundary": "do not promote until components validate",
        },
        {
            "candidate_improvement": "Native hp_activity timing/QC sidecar",
            "evidence_reviewed": "nb13_native_qc_sidecar.csv",
            "output_impact": "additive timing/QC support",
            "risk": "arbitrary-unit native sidecar, not velocity",
            "current_classification": "validated from raw native timing/QC bytes",
            "later_nbdev_boundary": "export as experimental native_qc helper",
        },
        {
            "candidate_improvement": "Sweep-marker mask",
            "evidence_reviewed": "nb13_delta_sweep_marker_mask.csv",
            "output_impact": "localized frame velocity changes",
            "risk": "may remove true signal near ROI edge",
            "current_classification": "needs visual review",
            "later_nbdev_boundary": "not default",
        },
        {
            "candidate_improvement": "Autocorrelation-guided beat detector",
            "evidence_reviewed": "nb13_delta_autocorr_beat_detector.csv",
            "output_impact": "changes peaks and complete-beat counts in selected recordings",
            "risk": "not morphology-validated",
            "current_classification": "candidate behavior measured",
            "later_nbdev_boundary": "keep candidate-labeled or notebook-only",
        },
        {
            "candidate_improvement": "Audio clipping candidate QC",
            "evidence_reviewed": "nb13_audio_clipping_candidate.csv",
            "output_impact": "additive audio QC flag",
            "risk": "threshold-dependent",
            "current_classification": "candidate for later modularization",
            "later_nbdev_boundary": "export as additive audio_qc helper",
        },
        {
            "candidate_improvement": "Audio/native HR agreement",
            "evidence_reviewed": "nb13_cross_modal_hr_agreement.csv",
            "output_impact": "additive recording-level QC",
            "risk": "timing estimate only; missing audio must not count as disagreement",
            "current_classification": "candidate for later modularization",
            "later_nbdev_boundary": "export classification helper if useful",
        },
    ]
)

final_checkpoint_csv_path = NB13_OUT_DIR / "nb13_final_checkpoint.csv"
final_checkpoint.to_csv(final_checkpoint_csv_path, index=False)

print("Final evidence-stage checkpoint")
print("-" * 88)
print(f"checkpoint rows:                   {len(final_checkpoint)}")
print(f"nbdev export candidate rows:        {len(nbdev_export_candidates)}")
print(f"saved checkpoint CSV:               {final_checkpoint_csv_path}")
print(f"saved nbdev candidates CSV:         {nbdev_candidates_csv_path}")
print(f"saved calibration summary CSV:      {NB13_OUT_DIR / 'nb13_flag_experiment_summary.csv'}")
print("-" * 88)

display(final_checkpoint)
display(nbdev_export_candidates)

Final evidence-stage checkpoint
----------------------------------------------------------------------------------------
checkpoint rows:                   12
nbdev export candidate rows:        9
saved checkpoint CSV:               D:\code\DopplerLab\feature_exports\nb13_validation\nb13_final_checkpoint.csv
saved nbdev candidates CSV:         D:\code\DopplerLab\feature_exports\nb13_validation\nb13_nbdev_export_candidates.csv
saved calibration summary CSV:      D:\code\DopplerLab\feature_exports\nb13_validation\nb13_flag_experiment_summary.csv
----------------------------------------------------------------------------------------


,candidate_improvement,evidence_reviewed,output_impact,risk,current_classification,later_nbdev_boundary
0,Baseline harness equals NB04 V2 frame output,nb13_baseline_reproduction_full.csv,reference only,none for validation anchor,validated in full-frame harness,export stable video/waveform/calibration helpers
1,Native-to-AVI mapping,file-size mapping from native linked AVI files,recording namespace bridge,duplicate-size ambiguity if future data collide,validated for current batch,export stable native_mapping helper
2,Native DcmRegionPara metadata parsing,nb13_native_metadata_calibration.csv,"metadata-derived ROI, baseline, scale, and tim...",metadata is not proof of better calibration,metadata-derived candidate input,"export parser, not defaults"
3,Native scale-only calibration,nb13_delta_native_scale_only.csv,small uniform velocity rescale,changes reported velocities,candidate for later modularization,"export calculation support, keep default uncha..."
4,Native ROI-only calibration,nb13_delta_native_roi_only.csv,localized ROI-edge velocity changes,may add or remove signal/artifact pixels,needs visual review,do not make default
5,Native baseline override,nb13_delta_native_baseline_override_only.csv a...,localized direction-aware re-reference,baseline shift changes all affected velocities,needs visual review,export helper logic only if kept clearly candi...
6,Native ROI plus scale plus baseline,nb13_delta_native_roi_scale_baseline.csv,combined metadata calibration delta,"compounds ROI, scale, and baseline effects","candidate, not default",do not promote until components validate
7,Native hp_activity timing/QC sidecar,nb13_native_qc_sidecar.csv,additive timing/QC support,"arbitrary-unit native sidecar, not velocity",validated from raw native timing/QC bytes,export as experimental native_qc helper
8,Sweep-marker mask,nb13_delta_sweep_marker_mask.csv,localized frame velocity changes,may remove true signal near ROI edge,needs visual review,not default
9,Autocorrelation-guided beat detector,nb13_delta_autocorr_beat_detector.csv,changes peaks and complete-beat counts in sele...,not morphology-validated,candidate behavior measured,keep candidate-labeled or notebook-only


,helper_or_group,planned_module,current_evidence,export_stage
0,find_repo_root / dopplerlab_project_paths,paths,path checks passed,stable helper candidate
1,read_video_frame_by_index_v2,video,single-frame and full-frame baseline reproduct...,stable helper candidate
2,extract_envelope_directional / extract_doppler...,waveform,full-frame baseline reproduction passed,stable helper candidate
3,fit_velocity_calibration_from_points / derive_...,calibration,baseline and split calibration deltas reproduced,stable helper candidate
4,build_native_to_avi_mapping,native_mapping,10/10 native recordings matched with no ambiguity,stable helper candidate
5,parse_dcm_region_pw,native_metadata,10/10 native metadata files parsed,stable helper candidate
6,native page/counter and hp_activity sidecar he...,native_qc,raw-byte sidecar reproduced and class agreemen...,experimental additive helper candidate
7,audio_clipping_metrics,audio_qc,registry clipping flag reproduced 9/9,experimental additive helper candidate
8,autocorr_guided_peaks,waveform,"candidate behavior measured, not visually vali...",keep candidate-labeled or notebook-only


### Interpretation - final evidence-stage checkpoint

**Result classification:** final evidence-stage checkpoint passed.

The checkpoint separates stable helper functions, candidate calibration changes, additive QC signals, and review-needed experimental behavior. The baseline harness, native-to-AVI mapping, native metadata parser, video reader, waveform helpers, and calibration helpers are the strongest candidates for later nbdev modularization.

The result supports using NB13 as the final validation bridge before package extraction.

The result does not support changing pipeline defaults, adopting native ROI or baseline override, replacing the beat detector, claiming native velocity-envelope recovery, or making clinical measurement claims.

The next working hypothesis is that the notebook run can close with a compact summary of generated artifacts, baseline reproduction, warnings, and nbdev export candidates.

## Run summary

### What this tests

This section summarizes the completed NB13 validation run, including generated CSV artifacts, baseline reproduction, native mapping warnings, missing-audio HR status, and nbdev export candidates.

### Why this matters

A compact run summary makes the notebook checkpoint easy to audit after reruns.


In [38]:
expected_csv_files = [
    "nb13_audio_clipping_candidate.csv",
    "nb13_baseline_override_diagnostics.csv",
    "nb13_baseline_reproduction_full.csv",
    "nb13_cross_modal_hr_agreement.csv",
    "nb13_delta_autocorr_beat_detector.csv",
    "nb13_delta_native_baseline_override_only.csv",
    "nb13_delta_native_roi_only.csv",
    "nb13_delta_native_roi_plus_scale.csv",
    "nb13_delta_native_roi_scale_baseline.csv",
    "nb13_delta_native_scale_only.csv",
    "nb13_delta_sweep_marker_mask.csv",
    "nb13_final_checkpoint.csv",
    "nb13_flag_experiment_summary.csv",
    "nb13_native_metadata_calibration.csv",
    "nb13_native_qc_sidecar.csv",
    "nb13_native_to_nb04_mapping.csv",
    "nb13_nbdev_export_candidates.csv",
]

generated_csv_files = sorted(path.name for path in NB13_OUT_DIR.glob("*.csv"))

missing_csv_files = sorted(set(expected_csv_files) - set(generated_csv_files))
unexpected_csv_files = sorted(set(generated_csv_files) - set(expected_csv_files))

unmatched_mapping_count = int((native_to_avi["match_status"] != "matched").sum())
not_assessed_hr_count = int(
    (cross_modal_hr_agreement["hr_agreement_status"] == "not_assessed_missing_audio").sum()
)
warning_count = unmatched_mapping_count + not_assessed_hr_count

baseline_n = len(baseline_reproduction)
baseline_median_abs_delta = float(baseline_reproduction["abs_delta_cm_s"].median())
baseline_max_abs_delta = float(baseline_reproduction["abs_delta_cm_s"].max())
baseline_exact_fraction = float((baseline_reproduction["abs_delta_cm_s"] < 0.01).mean())

print("NB13 run summary")
print("-" * 88)
print(f"generated CSV files:               {len(generated_csv_files)}")
print(f"expected CSV files:                {len(expected_csv_files)}")
print(f"missing expected CSV files:         {len(missing_csv_files)}")
print(f"unexpected CSV files:               {len(unexpected_csv_files)}")
print(f"output directory:                  {NB13_OUT_DIR}")
print(f"baseline reproduction rows:         {baseline_n}")
print(f"baseline median |delta| cm/s:       {baseline_median_abs_delta:.12f}")
print(f"baseline max |delta| cm/s:          {baseline_max_abs_delta:.12e}")
print(f"baseline exact fraction <0.01:      {baseline_exact_fraction:.3f}")
print(f"native mapping unmatched:           {unmatched_mapping_count}")
print(f"missing-audio HR not assessed:      {not_assessed_hr_count}")
print(f"warning count:                      {warning_count}")
print(f"nbdev export candidate rows:        {len(nbdev_export_candidates)}")
print("-" * 88)

display(pd.DataFrame({"generated_csv_file": generated_csv_files}))

if missing_csv_files:
    raise ValueError(f"Missing expected CSV files: {missing_csv_files}")

if unexpected_csv_files:
    raise ValueError(f"Unexpected CSV files found: {unexpected_csv_files}")

if baseline_n != 406:
    raise ValueError(f"Expected 406 baseline rows, found {baseline_n}")

if baseline_exact_fraction != 1.0:
    raise ValueError("Baseline exact fraction <0.01 cm/s was not 1.000")

if unmatched_mapping_count != 0:
    raise ValueError(f"Expected 0 unmatched native mappings, found {unmatched_mapping_count}")

if not_assessed_hr_count != 2:
    raise ValueError(f"Expected 2 not-assessed HR rows, found {not_assessed_hr_count}")

if len(nbdev_export_candidates) != 9:
    raise ValueError(f"Expected 9 nbdev export candidate rows, found {len(nbdev_export_candidates)}")

NB13 run summary
----------------------------------------------------------------------------------------
generated CSV files:               17
expected CSV files:                17
missing expected CSV files:         0
unexpected CSV files:               0
output directory:                  D:\code\DopplerLab\feature_exports\nb13_validation
baseline reproduction rows:         406
baseline median |delta| cm/s:       0.000000000000
baseline max |delta| cm/s:          1.421085471520e-14
baseline exact fraction <0.01:      1.000
native mapping unmatched:           0
missing-audio HR not assessed:      2
warning count:                      2
nbdev export candidate rows:        9
----------------------------------------------------------------------------------------


,generated_csv_file
0,nb13_audio_clipping_candidate.csv
1,nb13_baseline_override_diagnostics.csv
2,nb13_baseline_reproduction_full.csv
3,nb13_cross_modal_hr_agreement.csv
4,nb13_delta_autocorr_beat_detector.csv
5,nb13_delta_native_baseline_override_only.csv
6,nb13_delta_native_roi_only.csv
7,nb13_delta_native_roi_plus_scale.csv
8,nb13_delta_native_roi_scale_baseline.csv
9,nb13_delta_native_scale_only.csv


## Final NB13 checkpoint

**Result classification:** NB13 validation checkpoint completed.

The notebook reproduced the NB04 V2 AVI image-path baseline on all `406` frame rows, with only floating-point-level differences. Native-to-AVI mapping succeeded for all 10 native recordings with no unmatched or ambiguous mappings. Native `DcmRegionPara` metadata was parsed for all mapped recordings. Native `hp_activity` timing/QC sidecar features were reproduced from raw `PW_CinePartition0.bin` bytes and matched the current project native-specific classification boundary for this batch.

The split calibration experiments show that native scale-only produces a small uniform candidate velocity shift, native ROI-only produces localized ROI-edge effects, native baseline override produces localized direction-aware re-reference effects, and the combined native calibration candidate compounds these behaviors. These findings support native metadata as a candidate calibration support source, but they do not support changing the baseline pipeline default.

The robustness experiments show measurable candidate behavior for sweep-marker masking and autocorrelation-guided beat detection, but both require visual review before trust. Audio clipping QC reproduced the registry clipping flag for all evaluated recordings. Audio/native HR agreement provides an additive timing/QC comparison, with missing audio handled as `not_assessed_missing_audio`.

The nbdev export candidate table identifies stable helper groups for later modularization: path helpers, video frame reading, waveform extraction, calibration helpers, native mapping, and native metadata parsing. Native timing/QC and audio QC helpers remain experimental additive candidates. Candidate detector behavior should remain candidate-labeled or notebook-only until visually reviewed.

This notebook does not claim clinical validation, native spectrogram recovery, native velocity-envelope recovery, native PSV, EDV, RI, PI, VTI, or production-ready Doppler measurements.